# Neural Local Volatility v7 — final raw revision

This is the final Neural-LV notebook before moving to Neural SLV.

The revision deliberately removes mechanisms that were added to protect against hypothetical problems but were not justified by the March-2024 SPX data. The core model remains the best-performing v6 specification:

- **raw price-space calibration**;
- **shared 4×50 ReLU trunk + maturity-specific Softplus heads**;
- **Sabaté-style learned control variates**;
- corrected fixed-contract spot delta with common random numbers;
- correct absolute model clock for future-date pricing.

The main changes are methodological rather than another architecture search:

1. **All filtered OTM observations are eligible for calibration** — the 25-quotes-per-slice cap is removed by default.
2. Training is **plain contract-level raw MSE**. There is no maturity-balanced sampler, no maturity-balanced loss and no maturity-balanced checkpoint criterion.
3. Rolling training is counted in **SDE update phases**, not alternating nominal epochs.
4. The fixed-path **raw RMSE monitor is evaluated after every SDE phase**. Early stopping is based directly on consecutive SDE updates without improvement.
5. The best raw-RMSE candidates are reranked independently at 50k paths.
6. Rolling warm-start transfer corrects the trunk's time input exactly by shifting the first-layer bias, while maturity heads are mapped to the same calendar expiries.
7. The 80/20 split is used only to measure same-day interpolation. After that measurement, the model is **polished on 100% of the current day's OTM surface** and that full-data model is used for next-day forecasting and as the next day's warm start.
8. **OTM-on-both-dates next-day pricing is the primary temporal metric**. The full clean chain remains a secondary extrapolation metric.
9. The same one-day hedge diagnostic is reported with both the Neural-LV delta and the OptionMetrics market delta.

The March-1 hold-out remains a **validation/audit** result rather than a pristine test because the inherited v5/v6 initialization participated in earlier model development. Later-date hold-outs are the cleaner evidence after this specification is frozen.

No LR scheduler is used. Classical BS/LV/Heston comparison remains deferred to the final common benchmark notebook after NSLV.


### Crash-safe execution changes

This build is sized for a 16-GB Windows/VS Code machine. Final evaluation uses 2,000-path chunks, preallocated float32 terminal arrays, and AM/PM settlement groups are simulated and priced sequentially rather than being held in memory together.

The untouched warm-start model is also checkpoint candidate **SDE update 0**. This prevents the 100%-data polish from being forced to accept a deterioration if every gradient update is worse than the starting model.


## Crash diagnostic revision
The cause of the native kernel exit is not established. This revision adds persistent stage/RSS logs, environment details and Python fault handling, replaces only the summary Pearson-correlation implementation, and saves the first transition immediately. Training, model, path counts and existing cache location are retained.
Restart the kernel, then run in order in the original project folder. Keep `FORCE_RETRAIN=False` and `neural_lv_v7_final_safe_cache` to reuse completed training. If another crash occurs, inspect `v7_stage_trace.log`, `v7_native_fault.log`, `v7_environment.txt` and VS Code Jupyter output. An empty native log does not rule out an OS-level kill.


## 1. Setup and locked final-v7 configuration

In [1]:
from pathlib import Path
from dataclasses import dataclass
import copy, gc, json, math, os, random, warnings

# Prior versions had native-kernel instability on this laptop, so CPU mode stays conservative.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from scipy.optimize import brentq
from scipy.special import ndtr
from tqdm import tqdm
import torch
import torch.nn as nn

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 12345
ANCHOR_DATE = pd.Timestamp("2024-03-01")
COMPUTE_PROFILE = "cpu"          # "cpu" or "gpu_hpc"
FORCE_RETRAIN = False
RUN_SEED_ROBUSTNESS = False      # optional; keep off for the main overnight run

HELDOUT_FRACTION = 0.20
USE_ALL_FILTERED_OTM = True      # final-v7 default: no 25-per-slice calibration cap
MAX_QUOTES_PER_SLICE_FALLBACK = 25
BASE_STEPS_PER_YEAR = 96

# Locked Neural-LV architecture.
SDE_HIDDEN_LAYERS, SDE_WIDTH = 4, 50
HEDGE_HIDDEN_LAYERS, HEDGE_WIDTH = 3, 30

# Raw rolling calibration. No LR scheduler.
SDE_LR = 1e-4
HEDGE_LR = 1e-3
PATHS_PER_STEP = 4_096
STEPS_PER_PHASE = 10
CONTRACTS_PER_STEP = 400

# Main 80%-fit budget is counted in actual SDE phases.
MAX_SDE_UPDATES = 100
EARLY_MIN_SDE_UPDATES = 20
EARLY_PATIENCE_SDE_UPDATES = 10

# After the untouched 20% has been evaluated, fit all current-day OTM quotes
# before next-day forecasting / passing weights to tomorrow.
POLISH_MAX_SDE_UPDATES = 30
POLISH_EARLY_MIN_SDE_UPDATES = 10
POLISH_EARLY_PATIENCE_SDE_UPDATES = 12

# Fixed-path training-only monitor and independent checkpoint rerank.
MONITOR_PATHS = 16_384
TOP_K_CHECKPOINTS = 10
RERANK_PATHS = 50_000
RESUME_SAVE_EVERY_SDE = 5

# Final evaluation.
FINAL_MC_PATHS = 100_000
FINAL_PATH_CHUNK = 2_000
DELTA_BUMP_BPS = (1, 5, 10)
DELTA_HEADLINE_BPS = 5

# CPU: five transitions. GPU/HPC: entire loaded month.
MAX_TRANSITIONS = 5 if COMPUTE_PROFILE == "cpu" else None

CACHE_DIR = Path("neural_lv_v7_final_safe_cache")
V6_CACHE_DIR = Path("neural_lv_v6_cache")
CACHE_DIR.mkdir(exist_ok=True)

def choose_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch, "xpu"):
        try:
            if torch.xpu.is_available():
                return torch.device("xpu")
        except Exception:
            pass
    return torch.device("cpu")

DEVICE = choose_device()
if DEVICE.type == "cpu":
    torch.set_num_threads(1)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass

def set_seed(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

set_seed(SEED)

print("Device:", DEVICE, "| PyTorch threads:", torch.get_num_threads())
print("Compute profile:", COMPUTE_PROFILE, "| transitions:", MAX_TRANSITIONS or "all available")
print("Calibration universe:", "all filtered OTM" if USE_ALL_FILTERED_OTM else "capped OTM")
print("Objective: plain contract-level raw price MSE (spot-scaled)")
print("Main budget:", MAX_SDE_UPDATES, "SDE phases | polish:", POLISH_MAX_SDE_UPDATES, "SDE phases")
print("LR:", SDE_LR, "| hedge LR:", HEDGE_LR, "| scheduler: none")
print("Early stop:", EARLY_PATIENCE_SDE_UPDATES, "SDE phases without a new raw-RMSE best")

# Persistent diagnostics: keep the fault-handler file open for the kernel lifetime.
import sys, platform, faulthandler, traceback
from datetime import datetime
try:
    import psutil as _psutil
except ImportError:
    _psutil = None
if "_nlv_fault_file" in globals():
    faulthandler.disable()
    _nlv_fault_file.close()
_nlv_fault_file = open(CACHE_DIR / "v7_native_fault.log", "a", buffering=1)
faulthandler.enable(file=_nlv_fault_file, all_threads=True)

def stage_log(message):
    memory = "RSS unavailable (psutil not installed)"
    if _psutil is not None:
        try:
            memory = f"RSS={_psutil.Process().memory_info().rss / 2**20:.1f} MiB"
        except Exception as exc:
            memory = f"RSS unavailable: {type(exc).__name__}"
    line = f"{datetime.now().isoformat(timespec='seconds')} | {message} | {memory}"
    print(line, flush=True)
    with open(CACHE_DIR / "v7_stage_trace.log", "a", encoding="utf-8") as handle:
        handle.write(line + "\n")
        handle.flush()

stage_log(f"New session: Python={sys.version}; executable={sys.executable}; "
          f"platform={platform.platform()}; numpy={np.__version__}; "
          f"pandas={pd.__version__}; torch={torch.__version__}")
with open(CACHE_DIR / "v7_environment.txt", "w", encoding="utf-8") as handle:
    import contextlib
    with contextlib.redirect_stdout(handle):
        print(sys.version, sys.executable, platform.platform())
        print("numpy", np.__version__, "pandas", pd.__version__, "torch", torch.__version__)
        np.show_config()
        print(torch.__config__.show())
        try:
            from threadpoolctl import threadpool_info
            print(threadpool_info())
        except ImportError:
            print("threadpoolctl unavailable")


Device: cpu | PyTorch threads: 1
Compute profile: cpu | transitions: 5
Calibration universe: all filtered OTM
Objective: plain contract-level raw price MSE (spot-scaled)
Main budget: 100 SDE phases | polish: 30 SDE phases
LR: 0.0001 | hedge LR: 0.001 | scheduler: none
Early stop: 10 SDE phases without a new raw-RMSE best
2026-09-21T09:55:41 | New session: Python=3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]; executable=c:\Users\Startklar\anaconda3.0\python.exe; platform=Windows-11-10.0.26200-SP0; numpy=1.26.4; pandas=2.2.2; torch=2.9.1+cpu | RSS=295.9 MiB


## 2. Load and clean OptionMetrics data


In [2]:
def find_file(pattern):
    search_dirs = [Path("."), Path("/mnt/data")]
    found = []
    for folder in search_dirs:
        found.extend(sorted(folder.glob(pattern)))
    if not found:
        raise FileNotFoundError(f"Could not find a file matching {pattern!r}")
    return found[0]


FILES = {
    "options": find_file("one_month_spx_option_price.csv"),
    "forwards": find_file("one_month_spx_forward_price.csv"),
    "spot": find_file("one_month_spx_security_price.csv"),
    "rates": find_file("one_month_spx_zero_coupon_yield_curve.csv"),
}

for name, path in FILES.items():
    print(f"{name:8s}: {path}")

options_raw = pd.read_csv(FILES["options"], low_memory=False)
forwards_raw = pd.read_csv(FILES["forwards"], low_memory=False)
spot_raw = pd.read_csv(FILES["spot"], low_memory=False)
rates_raw = pd.read_csv(FILES["rates"], low_memory=False)



SECID = 108105
MIN_DTE, MAX_DTE = 7, 365
MAX_ABS_LOG_MONEYNESS = 0.30
MIN_ABS_DELTA, MAX_ABS_DELTA = 0.05, 0.95
MAX_RELATIVE_SPREAD = 0.25
ATM_TIGHTER_SIDE_BAND = 0.05
MIN_POINTS_PER_SLICE = 8


def prepare_clean_data(options_raw, forwards_raw, spot_raw, rates_raw):
    options = options_raw.copy()
    options["date"] = pd.to_datetime(options["date"])
    options["exdate"] = pd.to_datetime(options["exdate"])
    options["K"] = options["strike_price"] / 1000.0
    options["T_days"] = (options["exdate"] - options["date"]).dt.days
    options["T"] = options["T_days"] / 365.0
    options["bid"] = pd.to_numeric(options["best_bid"], errors="coerce")
    options["ask"] = pd.to_numeric(options["best_offer"], errors="coerce")
    options["mid"] = 0.5 * (options["bid"] + options["ask"])
    options["spread"] = options["ask"] - options["bid"]
    options["relative_spread"] = options["spread"] / options["mid"].replace(0, np.nan)
    options["iv_market"] = pd.to_numeric(options["impl_volatility"], errors="coerce")
    options["delta_market"] = pd.to_numeric(options["delta"], errors="coerce")
    options["abs_delta_market"] = options["delta_market"].abs()

    mask = (
        (options["secid"] == SECID)
        & (options["exercise_style"] == "E")
        & (options["contract_size"] == 100)
        & (options["cfadj"] == 1)
        & options["cp_flag"].isin(["C", "P"])
        & (options["bid"] >= 0)
        & (options["ask"] > options["bid"])
        & (options["mid"] > 0)
        & options["T_days"].between(MIN_DTE, MAX_DTE)
        & options["iv_market"].between(1e-4, 5.0)
        & options["abs_delta_market"].between(MIN_ABS_DELTA, MAX_ABS_DELTA)
        & (options["relative_spread"] <= MAX_RELATIVE_SPREAD)
    )
    options = options.loc[mask].copy()

    spot = spot_raw.loc[spot_raw["secid"] == SECID, ["date", "close"]].copy()
    spot["date"] = pd.to_datetime(spot["date"])
    spot = spot.rename(columns={"close": "S"}).drop_duplicates("date")

    forwards = forwards_raw.loc[forwards_raw["secid"] == SECID].copy()
    forwards["date"] = pd.to_datetime(forwards["date"])
    forwards["expiration"] = pd.to_datetime(forwards["expiration"])
    forwards = forwards.rename(columns={
        "expiration": "exdate",
        "AMSettlement": "am_settlement",
        "ForwardPrice": "F",
    })
    forwards = forwards[
        ["date", "exdate", "am_settlement", "F"]
    ].drop_duplicates(["date", "exdate", "am_settlement"])

    rates = rates_raw.copy()
    rates["date"] = pd.to_datetime(rates["date"])
    rates["days"] = pd.to_numeric(rates["days"], errors="coerce")
    rates["rate_decimal"] = pd.to_numeric(rates["rate"], errors="coerce") / 100.0
    rate_curves = {
        d: (g["days"].to_numpy(float), g["rate_decimal"].to_numpy(float))
        for d, g in rates.dropna().sort_values(["date", "days"]).groupby("date")
    }

    options = options.merge(spot, on="date", how="left").merge(
        forwards,
        on=["date", "exdate", "am_settlement"],
        how="left",
        validate="many_to_one",
    )
    options["r"] = [
        float(np.interp(n, *rate_curves[d])) if d in rate_curves else np.nan
        for d, n in zip(options["date"], options["T_days"])
    ]
    options["D"] = np.exp(-options["r"] * options["T"])
    options["carry"] = np.log(options["F"] / options["S"]) / options["T"]
    options["q"] = options["r"] - options["carry"]
    options["k"] = np.log(options["K"] / options["F"])

    options = options.dropna(subset=["S", "F", "r", "D", "carry", "k"])
    options = options.loc[options["k"].abs() <= MAX_ABS_LOG_MONEYNESS].copy()

    call_lb = options["D"] * np.maximum(options["F"] - options["K"], 0.0)
    call_ub = options["D"] * options["F"]
    put_lb = options["D"] * np.maximum(options["K"] - options["F"], 0.0)
    put_ub = options["D"] * options["K"]
    lb = np.where(options["cp_flag"].eq("C"), call_lb, put_lb)
    ub = np.where(options["cp_flag"].eq("C"), call_ub, put_ub)
    options = options.loc[
        (options["mid"] >= lb - 1e-6) & (options["mid"] <= ub + 1e-6)
    ].copy()

    return options.sort_values(
        ["date", "exdate", "am_settlement", "K", "cp_flag"]
    ).reset_index(drop=True)


def build_otm_surface(quotes):
    idx = [
        "date", "exdate", "am_settlement", "K", "T", "S", "F",
        "r", "q", "carry", "D", "k"
    ]
    vals = [
        "mid", "bid", "ask", "spread", "relative_spread",
        "iv_market", "delta_market", "optionid"
    ]

    wide = quotes.pivot_table(
        index=idx, columns="cp_flag", values=vals, aggfunc="first"
    )
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()

    strict_call = wide["K"] >= wide["F"]
    near = wide["k"].abs() <= ATM_TIGHTER_SIDE_BAND
    have_c = wide.get("iv_market_C", pd.Series(np.nan, index=wide.index)).notna()
    have_p = wide.get("iv_market_P", pd.Series(np.nan, index=wide.index)).notna()
    c_spread = wide.get("spread_C", pd.Series(np.inf, index=wide.index))
    p_spread = wide.get("spread_P", pd.Series(np.inf, index=wide.index))

    choose_call = strict_call.copy()
    both_near = near & have_c & have_p
    choose_call = np.where(both_near, c_spread <= p_spread, choose_call)
    wide["cp_otm"] = np.where(choose_call, "C", "P")

    for base in vals:
        c = wide.get(f"{base}_C", pd.Series(np.nan, index=wide.index))
        p = wide.get(f"{base}_P", pd.Series(np.nan, index=wide.index))
        wide[f"{base}_otm"] = np.where(choose_call, c, p)

    wide = wide.dropna(subset=["iv_market_otm", "mid_otm", "optionid_otm"]).copy()

    sizes = wide.groupby(["date", "exdate", "am_settlement"]).size()
    valid = sizes[sizes >= MIN_POINTS_PER_SLICE].index

    return (
        wide.set_index(["date", "exdate", "am_settlement"])
        .loc[valid]
        .reset_index()
        .sort_values(["date", "T", "k"])
        .reset_index(drop=True)
    )


quotes_all = prepare_clean_data(options_raw, forwards_raw, spot_raw, rates_raw)
otm = build_otm_surface(quotes_all)

print("Complete cleaned quotes:", len(quotes_all))
print("OTM calibration observations:", len(otm))
print("Trading dates:", otm["date"].nunique())
print("Expiry/settlement slices:", otm.groupby(["date", "exdate", "am_settlement"]).ngroups)


options : one_month_spx_option_price.csv
forwards: one_month_spx_forward_price.csv
spot    : one_month_spx_security_price.csv
rates   : one_month_spx_zero_coupon_yield_curve.csv
Complete cleaned quotes: 192031
OTM calibration observations: 91930
Trading dates: 20
Expiry/settlement slices: 809


## 3. Curves, selected surface and deterministic daily 80/20 split


In [3]:
@dataclass
class DailyMarketCurve:
    date: pd.Timestamp
    am_settlement: int
    spot: float
    T_nodes: np.ndarray
    logD_nodes: np.ndarray
    logF_nodes: np.ndarray

    def __post_init__(self):
        self._logD = PchipInterpolator(self.T_nodes, self.logD_nodes, extrapolate=True)
        self._logF = PchipInterpolator(self.T_nodes, self.logF_nodes, extrapolate=True)

    def log_discount(self, t):
        return self._logD(np.maximum(np.asarray(t, float), 0.0))

    def log_forward_ratio(self, t):
        return self._logF(np.maximum(np.asarray(t, float), 0.0))

    def discount(self, t0, t1):
        return np.exp(self.log_discount(t1) - self.log_discount(t0))

    def forward(self, t):
        return self.spot * np.exp(self.log_forward_ratio(t))


class CurveBook:
    def __init__(self, spot_raw, forwards_raw, rates_raw):
        spot = spot_raw.loc[spot_raw["secid"] == SECID, ["date", "close"]].copy()
        spot["date"] = pd.to_datetime(spot["date"])
        self.spot = spot.set_index("date")["close"].to_dict()

        self.forwards = forwards_raw.loc[forwards_raw["secid"] == SECID].copy()
        self.forwards["date"] = pd.to_datetime(self.forwards["date"])
        self.forwards["expiration"] = pd.to_datetime(self.forwards["expiration"])

        self.rates = rates_raw.copy()
        self.rates["date"] = pd.to_datetime(self.rates["date"])
        self.rates["rate_decimal"] = pd.to_numeric(self.rates["rate"], errors="coerce") / 100.0
        self._cache = {}

    def get(self, date, am_settlement):
        date, am = pd.Timestamp(date), int(am_settlement)
        key = (date, am)
        if key in self._cache:
            return self._cache[key]

        S0 = float(self.spot[date])
        rg = self.rates.loc[self.rates["date"] == date, ["days", "rate_decimal"]].dropna().sort_values("days")
        T_r = rg["days"].to_numpy(float) / 365.0
        logD = -rg["rate_decimal"].to_numpy(float) * T_r

        fg = self.forwards.loc[
            (self.forwards["date"] == date) & (self.forwards["AMSettlement"] == am),
            ["expiration", "ForwardPrice"],
        ].copy()
        if len(fg) < 2:
            fg = self.forwards.loc[self.forwards["date"] == date, ["expiration", "ForwardPrice"]].copy()

        fg["T"] = (fg["expiration"] - date).dt.days / 365.0
        fg = (
            fg.loc[(fg["T"] >= 0) & (fg["ForwardPrice"] > 0)]
            .groupby("T", as_index=False)["ForwardPrice"].mean().sort_values("T")
        )
        T_f = fg["T"].to_numpy(float)
        logF = np.log(fg["ForwardPrice"].to_numpy(float) / S0)
        positive = T_f > 0
        T_f, logF = T_f[positive], logF[positive]

        T_all = np.unique(np.concatenate([[0.0], T_r, T_f]))
        logD_all = np.interp(T_all, np.r_[0.0, T_r], np.r_[0.0, logD])
        logF_all = np.interp(T_all, np.r_[0.0, T_f], np.r_[0.0, logF])

        curve = DailyMarketCurve(date, am, S0, T_all, logD_all, logF_all)
        self._cache[key] = curve
        return curve


curve_book = CurveBook(spot_raw, forwards_raw, rates_raw)


def sample_calibration_day(day, max_per_slice=MAX_QUOTES_PER_SLICE_FALLBACK):
    """Fallback CPU subsampler. The final-v7 default uses all filtered OTM quotes."""
    pieces = []
    for _, g in day.groupby(["exdate", "am_settlement"]):
        g = g.sort_values("k")
        if max_per_slice is None or len(g) <= max_per_slice:
            pieces.append(g)
        else:
            pos = np.unique(np.round(np.linspace(0, len(g)-1, max_per_slice)).astype(int))
            pieces.append(g.iloc[pos])
    return pd.concat(pieces, ignore_index=True)


def prepare_selected_universe(date):
    day = otm.loc[otm["date"] == pd.Timestamp(date)].copy()
    base = day if USE_ALL_FILTERED_OTM else sample_calibration_day(day)
    selected = base.rename(columns={
        "mid_otm": "target_mid",
        "cp_otm": "target_cp",
        "optionid_otm": "optionid",
    }).reset_index(drop=True)
    selected["full_selected_id"] = np.arange(len(selected), dtype=int)
    return selected

def stratified_holdout(selected, fraction=HELDOUT_FRACTION, seed=SEED+77):
    """~80/20 within each expiry/settlement slice, spread across moneyness."""
    selected = selected.copy().reset_index(drop=True)
    is_holdout = np.zeros(len(selected), dtype=bool)
    rng = np.random.default_rng(seed)
    stride = max(2, int(round(1.0 / fraction)))

    for _, g in selected.groupby(["exdate", "am_settlement"], sort=False):
        ordered = g.sort_values("k").index.to_numpy()
        chosen = ordered[int(rng.integers(0, stride))::stride]
        if len(chosen) == 0:
            chosen = ordered[[len(ordered)//2]]
        if len(chosen) >= len(ordered):
            chosen = chosen[:-1]
        is_holdout[chosen] = True

    selected["split"] = np.where(is_holdout, "holdout", "train")
    train = selected.loc[~is_holdout].copy().reset_index(drop=True)
    holdout = selected.loc[is_holdout].copy().reset_index(drop=True)
    train["contract_id"] = np.arange(len(train), dtype=int)
    return selected, train, holdout


def daily_split(date):
    selected = prepare_selected_universe(date)
    return stratified_holdout(selected)


trading_dates = sorted(pd.to_datetime(quotes_all["date"].unique()))
if ANCHOR_DATE not in trading_dates:
    raise ValueError(f"Anchor date {ANCHOR_DATE.date()} is not in the option dataset.")

anchor_selected, anchor_train, anchor_holdout = daily_split(ANCHOR_DATE)
print("Anchor selected/train/validation:", len(anchor_selected), len(anchor_train), len(anchor_holdout))
print("Anchor selected as fraction of filtered OTM:", f"{len(anchor_selected)/len(otm.loc[otm['date']==ANCHOR_DATE]):.1%}")
print("Anchor maturities train/validation:", anchor_train["T"].nunique(), anchor_holdout["T"].nunique())


Anchor selected/train/validation: 4304 3443 861
Anchor selected as fraction of filtered OTM: 100.0%
Anchor maturities train/validation: 36 36


## 4. Locked maturity-head Neural LV and warm-start rules

The trunk is shared across all maturities. Each time interval has a small `50 -> 1 -> Softplus` volatility head.

For rolling recalibration from date $d$ to $d'$:

- copy the trunk exactly;
- a new head ending at $T_{\text{new}}$ inherits the old head nearest to
  $T_{\text{new}} + (d'-d)/365$, approximating the same calendar expiry;
- the control-variate hedge bank is new each day because the option identifiers/output dimension change.

March 1 is loaded from the completed v6 **raw-loss maturity-head** checkpoint. It is not retrained in the main v7 run.


In [4]:
class FeedForwardTrunk(nn.Module):
    def __init__(self, input_dim=2, hidden_layers=SDE_HIDDEN_LAYERS, width=SDE_WIDTH):
        super().__init__()
        layers, d = [], input_dim
        for _ in range(hidden_layers):
            layers += [nn.Linear(d, width), nn.ReLU()]
            d = width
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def sabate_weight_initialisation(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_normal_(module.weight.data, gain=1.5)


def _tx(t, x):
    if x.ndim == 1:
        x = x[:, None]
    if not torch.is_tensor(t):
        t = torch.full_like(x, float(t))
    elif t.ndim == 0:
        t = torch.full_like(x, float(t.item()))
    elif t.ndim == 1:
        t = t[:, None]
    return torch.cat([t, x], dim=1)


class MaturityHeadNeuralLocalVol(nn.Module):
    def __init__(self, maturity_boundaries):
        super().__init__()
        self.boundaries = np.asarray(maturity_boundaries, float)
        self.trunk = FeedForwardTrunk()
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(SDE_WIDTH, 1), nn.Softplus())
            for _ in self.boundaries
        ])
        self.apply(sabate_weight_initialisation)

    def interval_index(self, t):
        tt = float(t.reshape(-1)[0].item()) if torch.is_tensor(t) else float(t)
        idx = int(np.searchsorted(self.boundaries, tt + 1e-7, side="right"))
        return min(idx, len(self.heads)-1)

    def forward(self, t, x):
        return self.heads[self.interval_index(t)](self.trunk(_tx(t, x)))


class PerOptionHedgeNet(nn.Module):
    def __init__(self, n_options):
        super().__init__()
        self.trunk = FeedForwardTrunk(2, HEDGE_HIDDEN_LAYERS, HEDGE_WIDTH)
        self.output = nn.Linear(HEDGE_WIDTH, int(n_options))
        self.apply(sabate_weight_initialisation)

    def forward_ids(self, t, x, option_ids):
        h = self.trunk(_tx(t, x))
        ids = torch.as_tensor(option_ids, dtype=torch.long, device=h.device)
        return h @ self.output.weight[ids].T + self.output.bias[ids][None, :]


class IntervalHedgeBank(nn.Module):
    def __init__(self, maturity_boundaries, n_options):
        super().__init__()
        self.boundaries = np.asarray(maturity_boundaries, float)
        self.nets = nn.ModuleList([
            PerOptionHedgeNet(n_options) for _ in self.boundaries
        ])

    def interval_index(self, t):
        tt = float(t.reshape(-1)[0].item()) if torch.is_tensor(t) else float(t)
        idx = int(np.searchsorted(self.boundaries, tt + 1e-7, side="right"))
        return min(idx, len(self.nets)-1)

    def forward_ids(self, t, x, option_ids):
        return self.nets[self.interval_index(t)].forward_ids(t, x, option_ids)


def set_requires_grad(module, value):
    for p in module.parameters():
        p.requires_grad_(value)


def safe_torch_load(path, map_location=DEVICE):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def transfer_rolling_model(previous, current, previous_date, current_date):
    """Warm-start by calendar time, including the trunk's explicit time input.

    The shared trunk receives [t, x]. Advancing the valuation date by elapsed means that
    the old function at the same calendar point is h_old(t + elapsed, x). Because the
    first trunk layer is affine, this shift is represented exactly by
        b_new = b_old + W_time * elapsed.
    Heads are mapped to the old boundary nearest T_new + elapsed.
    """
    elapsed = (pd.Timestamp(current_date) - pd.Timestamp(previous_date)).days / 365.0
    if elapsed < -1e-12:
        raise ValueError("Rolling transfer requires a non-decreasing calendar date.")

    current.trunk.load_state_dict(previous.trunk.state_dict())

    # Exact time-input shift in the first affine layer.
    first = current.trunk.net[0]
    if not isinstance(first, nn.Linear) or first.weight.shape[1] != 2:
        raise RuntimeError("Unexpected trunk first layer; cannot apply exact time shift.")
    if abs(elapsed) > 0:
        with torch.no_grad():
            first.bias.add_(first.weight[:, 0] * float(elapsed))

    matched = []
    with torch.no_grad():
        for i, T_new in enumerate(current.boundaries):
            target_old_T = float(T_new) + elapsed
            j = int(np.argmin(np.abs(previous.boundaries - target_old_T)))
            current.heads[i].load_state_dict(previous.heads[j].state_dict())
            matched.append(abs(float(previous.boundaries[j]) - target_old_T))

    print(
        f"Warm-start {pd.Timestamp(previous_date).date()} -> {pd.Timestamp(current_date).date()}: "
        f"{len(current.heads)} heads | elapsed={elapsed:.5f}y | "
        f"median head mismatch={np.median(matched):.5f}y"
    )


@torch.no_grad()
def trunk_time_shift_unit_test():
    prev = MaturityHeadNeuralLocalVol([0.10, 0.25, 0.50]).to(DEVICE).eval()
    cur = MaturityHeadNeuralLocalVol([0.09, 0.24, 0.49]).to(DEVICE).eval()
    d0, d1 = pd.Timestamp("2024-03-01"), pd.Timestamp("2024-03-04")
    transfer_rolling_model(prev, cur, d0, d1)
    elapsed = (d1-d0).days/365.0

    x = torch.linspace(0.8, 1.2, 31, device=DEVICE)[:, None]
    t = torch.full_like(x, 0.13)
    a = prev.trunk(_tx(t + elapsed, x))
    b = cur.trunk(_tx(t, x))
    err = float(torch.max(torch.abs(a-b)).cpu())
    if err > 2e-6:
        raise RuntimeError(f"Trunk time-shift transfer failed: max error {err:.3e}")
    print(f"Trunk time-shift unit test max error={err:.2e}")
    return err


def locate_v6_anchor(date=ANCHOR_DATE):
    stamp = f"{pd.Timestamp(date):%Y%m%d}"
    exact = V6_CACHE_DIR / f"real_{stamp}_split80_20_ablation_rawloss_maturity_heads_resume.pt"
    if exact.exists():
        return exact
    matches = sorted(V6_CACHE_DIR.glob(f"*{stamp}*ablation_rawloss*maturity_heads*resume.pt"))
    return matches[0] if matches else None


## 5. Calibration context, raw-price objective and Sabaté control variates


In [5]:
def build_time_grid(maturities, steps_per_year=BASE_STEPS_PER_YEAR):
    maturities = np.sort(np.unique(np.asarray(maturities, float)))
    max_T = float(maturities[-1])
    base = np.arange(0.0, max_T + 0.5/steps_per_year, 1.0/steps_per_year)
    grid = np.unique(np.r_[0.0, base[base <= max_T + 1e-12], maturities, max_T])
    grid.sort()
    return grid


def nearest_grid_indices(grid, times):
    grid, times = np.asarray(grid), np.asarray(times)
    idx = np.clip(np.searchsorted(grid, times), 0, len(grid)-1)
    left = np.maximum(idx-1, 0)
    idx = np.where(np.abs(grid[left]-times) < np.abs(grid[idx]-times), left, idx)
    if np.max(np.abs(grid[idx]-times)) > 1e-10:
        raise RuntimeError("A listed maturity is not represented exactly on the simulation grid.")
    return idx


@dataclass
class CalibrationContext:
    date: pd.Timestamp
    spot: float
    contracts: pd.DataFrame
    maturity_boundaries: np.ndarray
    grid_np: np.ndarray
    grid: torch.Tensor
    maturity_index: torch.Tensor
    stage_index: np.ndarray
    stage_tensor: torch.Tensor
    K: torch.Tensor
    F_T: torch.Tensor
    D_T: torch.Tensor
    target: torch.Tensor
    cp_sign: torch.Tensor
    am: torch.Tensor
    F_grid: torch.Tensor
    D_grid: torch.Tensor


def build_calibration_context(date, contracts):
    date = pd.Timestamp(date)
    contracts = contracts.copy().reset_index(drop=True)
    contracts["contract_id"] = np.arange(len(contracts), dtype=int)

    boundaries = np.sort(contracts["T"].unique())
    grid_np = build_time_grid(boundaries)
    maturity_idx = nearest_grid_indices(grid_np, contracts["T"].to_numpy(float))
    stage_map = {round(float(T), 12): i for i, T in enumerate(boundaries)}
    stage_index = np.array([stage_map[round(float(T), 12)] for T in contracts["T"]], dtype=int)

    F_grid = np.empty((2, len(grid_np)))
    D_grid = np.empty((2, len(grid_np)))
    for am in [0, 1]:
        curve = curve_book.get(date, am)
        F_grid[am] = curve.forward(grid_np)
        D_grid[am] = curve.discount(0.0, grid_np)

    return CalibrationContext(
        date=date,
        spot=float(contracts["S"].iloc[0]),
        contracts=contracts,
        maturity_boundaries=boundaries,
        grid_np=grid_np,
        grid=torch.tensor(grid_np, dtype=torch.float32, device=DEVICE),
        maturity_index=torch.tensor(maturity_idx, dtype=torch.long, device=DEVICE),
        stage_index=stage_index,
        stage_tensor=torch.tensor(stage_index, dtype=torch.long, device=DEVICE),
        K=torch.tensor(contracts["K"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        F_T=torch.tensor(contracts["F"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        D_T=torch.tensor(contracts["D"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        target=torch.tensor(contracts["target_mid"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        cp_sign=torch.tensor(np.where(contracts["target_cp"].eq("C"), 1.0, -1.0), dtype=torch.float32, device=DEVICE),
        am=torch.tensor(contracts["am_settlement"].to_numpy(int), dtype=torch.long, device=DEVICE),
        F_grid=torch.tensor(F_grid, dtype=torch.float32, device=DEVICE),
        D_grid=torch.tensor(D_grid, dtype=torch.float32, device=DEVICE),
    )


def antithetic_normals(total_paths, n_steps, seed):
    half = (total_paths + 1) // 2
    gen = torch.Generator(device="cpu")
    gen.manual_seed(int(seed))
    z = torch.randn(half, n_steps, generator=gen, dtype=torch.float32)
    return torch.cat([z, -z], dim=0)[:total_paths].to(DEVICE)


def empirical_payoff_mean(samples, strike_norm, cp_flag):
    x = np.sort(np.asarray(samples, float))
    n = len(x)
    prefix = np.r_[0.0, np.cumsum(x)]
    k = np.asarray(strike_norm, float)
    idx = np.searchsorted(x, k, side="right")
    below, above = prefix[idx], prefix[-1] - prefix[idx]
    call = (above - k*(n-idx)) / n
    put = (k*idx - below) / n
    return np.where(np.asarray(cp_flag) == "C", call, put)


class UniformContractSampler:
    """Uniform sampling from the empirical training contracts."""
    def __init__(self, ctx, batch_size, seed):
        self.n_total = len(ctx.contracts)
        self.batch_size = min(int(batch_size), self.n_total)
        self.rng = np.random.default_rng(seed)

    def draw(self):
        if self.batch_size >= self.n_total:
            return np.arange(self.n_total, dtype=int)
        return self.rng.choice(self.n_total, size=self.batch_size, replace=False)

def simulate_estimators(model, hedge, ctx, contract_ids, z, phase):
    ids_np = np.asarray(contract_ids, dtype=int)
    ids = torch.as_tensor(ids_np, dtype=torch.long, device=DEVICE)
    maturity_idx = ctx.maturity_index[ids]
    K, F_T, D_T = ctx.K[ids], ctx.F_T[ids], ctx.D_T[ids]
    cp, am = ctx.cp_sign[ids], ctx.am[ids]

    B, M = z.shape[0], len(ids_np)
    x = torch.ones((B, 1), dtype=torch.float32, device=DEVICE)
    cv = torch.zeros((B, M), dtype=torch.float32, device=DEVICE)
    raw = torch.zeros((B, M), dtype=torch.float32, device=DEVICE)

    for step in range(int(maturity_idx.max().item())):
        t = ctx.grid[step]
        dt = ctx.grid[step+1] - ctx.grid[step]
        sqrt_dt = torch.sqrt(dt)
        sigma = model(t, x) if phase == "sde" else model(t, x).detach()
        dW = sqrt_dt * z[:, step:step+1]

        active = torch.where(maturity_idx > step)[0]
        if len(active):
            active_ids = ids_np[active.detach().cpu().numpy()]
            h = hedge.forward_ids(t, x.detach(), active_ids)
            if phase == "sde":
                h = h.detach()
            stochastic_piece = x.detach() * sigma.detach() * dW
            discounted_forward = ctx.D_grid[am[active], step] * ctx.F_grid[am[active], step]
            cv[:, active] += h * stochastic_piece * discounted_forward[None, :]

        a = x * sigma
        x = x + a*dW / (1.0 + torch.abs(x.detach()*sigma.detach())*sqrt_dt)

        matured = torch.where(maturity_idx == step+1)[0]
        if len(matured):
            S_T = x * F_T[matured][None, :]
            strike = K[matured][None, :]
            call = torch.clamp(S_T-strike, min=0.0)
            put = torch.clamp(strike-S_T, min=0.0)
            raw[:, matured] = torch.where((cp[matured] > 0)[None, :], call, put) * D_T[matured][None, :]

    return raw, raw-cv


@torch.no_grad()
def simulate_terminal_x(model, ctx, z):
    x = torch.ones((z.shape[0], 1), dtype=torch.float32, device=DEVICE)
    stage_by_grid = {
        int(nearest_grid_indices(ctx.grid_np, [T])[0]): j
        for j, T in enumerate(ctx.maturity_boundaries)
    }
    terminal = {}
    for step in range(len(ctx.grid_np)-1):
        t = ctx.grid[step]
        dt = ctx.grid[step+1] - ctx.grid[step]
        sigma = model(t, x)
        dW = torch.sqrt(dt) * z[:, step:step+1]
        a = x * sigma
        x = x + a*dW / (1.0 + torch.abs(a)*torch.sqrt(dt))
        if step+1 in stage_by_grid:
            terminal[stage_by_grid[step+1]] = x.squeeze(1).detach().cpu().numpy().copy()
    return terminal


def selected_prices_from_terminal(ctx, terminal):
    rows = []
    for stage in range(len(ctx.maturity_boundaries)):
        ids = np.where(ctx.stage_index == stage)[0]
        g = ctx.contracts.iloc[ids]
        payoff = empirical_payoff_mean(
            terminal[stage],
            g["K"].to_numpy(float) / g["F"].to_numpy(float),
            g["target_cp"].to_numpy(),
        )
        price = g["D"].to_numpy(float) * g["F"].to_numpy(float) * payoff
        tmp = g[["contract_id", "optionid", "T", "k", "target_mid"]].copy()
        tmp["model_price"] = price
        tmp["raw_error"] = price - tmp["target_mid"].to_numpy(float)
        rows.append(tmp)
    return pd.concat(rows, ignore_index=True)


def objective_diagnostics(model, ctx, z_fixed):
    """Plain empirical raw-price diagnostics on every contract in ctx."""
    table = selected_prices_from_terminal(ctx, simulate_terminal_x(model, ctx, z_fixed))
    e = table["raw_error"].to_numpy(float)
    raw_rmse = float(np.sqrt(np.mean(e**2)))
    return {
        "raw_rmse": raw_rmse,
        "scaled_raw_rmse": raw_rmse / float(ctx.spot),
        "raw_mae": float(np.mean(np.abs(e))),
        "table": table,
    }


def choose_even_ids(n_total, n):
    if n is None or n >= n_total:
        return np.arange(n_total, dtype=int)
    return np.unique(np.round(np.linspace(0, n_total-1, n)).astype(int))


@torch.no_grad()
def cv_variance_ratio(model, hedge, ctx, n_contracts=64, paths=8_192, seed=SEED+9000):
    ids = choose_even_ids(len(ctx.contracts), n_contracts)
    z = antithetic_normals(paths, len(ctx.grid_np)-1, seed)
    raw, cv = simulate_estimators(model, hedge, ctx, ids, z, phase="sde")
    ratio = torch.median(
        cv.var(0, unbiased=False) / torch.clamp(raw.var(0, unbiased=False), min=1e-12)
    )
    out = float(ratio.cpu())
    del z, raw, cv
    gc.collect()
    return out


anchor_ctx = build_calibration_context(ANCHOR_DATE, anchor_train)
print("Anchor training contracts:", len(anchor_ctx.contracts))
print("Anchor maturity heads:", len(anchor_ctx.maturity_boundaries))
print("Anchor simulation steps:", len(anchor_ctx.grid_np)-1)


Anchor training contracts: 3443
Anchor maturity heads: 36
Anchor simulation steps: 129


## 6. Final rolling trainer — plain raw objective, every-SDE monitoring and functional early stopping

One **SDE update phase** means `STEPS_PER_PHASE` model-gradient steps. It is followed by one hedge/control-variate phase unless early stopping fires.

The fixed-path raw RMSE is evaluated after **every** SDE phase. A candidate therefore cannot be missed merely because it fell between sparse ten-epoch monitoring points.

Only the small Neural-LV `state_dict` is retained for the top raw-RMSE candidates; candidate hedge/optimizer states are not duplicated. A single resume file contains the current training state.


In [6]:
@dataclass(frozen=True)
class TrainConfig:
    max_sde_updates: int = MAX_SDE_UPDATES
    paths_per_step: int = PATHS_PER_STEP
    steps_per_phase: int = STEPS_PER_PHASE
    contracts_per_step: int = CONTRACTS_PER_STEP
    monitor_paths: int = MONITOR_PATHS
    top_k: int = TOP_K_CHECKPOINTS
    rerank_paths: int = RERANK_PATHS
    save_every_sde: int = RESUME_SAVE_EVERY_SDE
    sde_lr: float = SDE_LR
    hedge_lr: float = HEDGE_LR
    early_min_sde: int = EARLY_MIN_SDE_UPDATES
    early_patience_sde: int = EARLY_PATIENCE_SDE_UPDATES
    seed_offset: int = 0


def checkpoint_file(run_tag):
    return CACHE_DIR / f"{run_tag}_resume.pt"


def cpu_state_dict(module):
    return {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}


def save_training_state(path, next_sde_update, model, hedge, opt_sde, opt_hedge,
                        history, candidates, best_monitor, bad_sde,
                        completed=False, stop_reason=None,
                        final_model_state=None, rerank_records=None):
    """Write resumable in-progress state; keep completed checkpoints compact."""
    if completed:
        payload = {
            "version": "v7_final_raw_safe",
            "next_sde_update": int(next_sde_update),
            "completed": True,
            "stop_reason": stop_reason,
            "boundaries": np.asarray(model.boundaries, float),
            "final_model_state": final_model_state or cpu_state_dict(model),
            "history": history,
            "rerank_records": rerank_records,
        }
    else:
        payload = {
            "version": "v7_final_raw_safe",
            "next_sde_update": int(next_sde_update),
            "completed": False,
            "stop_reason": stop_reason,
            "model_state": model.state_dict(),
            "hedge_state": hedge.state_dict(),
            "opt_sde": opt_sde.state_dict(),
            "opt_hedge": opt_hedge.state_dict(),
            "history": history,
            "candidates": candidates,
            "best_monitor": float(best_monitor),
            "bad_sde": int(bad_sde),
            "boundaries": np.asarray(model.boundaries, float),
        }
    torch.save(payload, path)



def update_candidates(model, candidates, sde_update, raw_rmse, top_k):
    candidate = {
        "sde_update": int(sde_update),
        "monitor_raw_rmse": float(raw_rmse),
        "model_state": cpu_state_dict(model),
    }
    out = list(candidates) + [candidate]
    out.sort(key=lambda x: x["monitor_raw_rmse"])
    return out[:int(top_k)]


@torch.no_grad()
def rerank_candidates(ctx, candidates, cfg):
    if not candidates:
        raise RuntimeError("No SDE checkpoints are available for reranking.")

    z = antithetic_normals(
        cfg.rerank_paths, len(ctx.grid_np)-1, SEED + 5000 + cfg.seed_offset
    )
    temp = MaturityHeadNeuralLocalVol(ctx.maturity_boundaries).to(DEVICE).eval()
    rows = []

    for c in tqdm(candidates, desc="Independent raw-RMSE rerank", unit="model"):
        temp.load_state_dict(c["model_state"], strict=True)
        d = objective_diagnostics(temp, ctx, z)
        rows.append({
            "sde_update": int(c["sde_update"]),
            "monitor_raw_rmse": float(c["monitor_raw_rmse"]),
            "rerank_raw_rmse": float(d["raw_rmse"]),
        })

    table = pd.DataFrame(rows).sort_values(
        ["rerank_raw_rmse", "monitor_raw_rmse"]
    ).reset_index(drop=True)
    table["independent_rank"] = np.arange(1, len(table)+1)

    best_update = int(table.iloc[0]["sde_update"])
    best = next(c for c in candidates if int(c["sde_update"]) == best_update)

    del z, temp
    gc.collect()
    return table, best


def run_training_phase(model, hedge, ctx, sampler, optimizer, phase, cfg):
    set_requires_grad(model, phase == "sde")
    set_requires_grad(hedge, phase == "hedge")
    losses = []

    for _ in range(cfg.steps_per_phase):
        ids = sampler.draw()
        z = torch.randn(
            cfg.paths_per_step, len(ctx.grid_np)-1,
            dtype=torch.float32, device=DEVICE
        )

        optimizer.zero_grad(set_to_none=True)
        raw, cv = simulate_estimators(model, hedge, ctx, ids, z, phase)
        ids_t = torch.as_tensor(ids, dtype=torch.long, device=DEVICE)

        if phase == "sde":
            # Spot is constant within a date, so this has the same minimiser as
            # unscaled raw contract-level price MSE while keeping gradients numerically tame.
            residual = (cv.mean(0) - ctx.target[ids_t]) / ctx.spot
            loss = residual.square().mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        else:
            variance = torch.var(cv / ctx.spot, dim=0, unbiased=False)
            loss = variance.mean()
            loss.backward()
            nn.utils.clip_grad_norm_(hedge.parameters(), 3.0)

        optimizer.step()
        losses.append(float(loss.detach().cpu()))
        del raw, cv, z

    return float(np.mean(losses))


def train_raw_day(ctx, cfg, run_tag, previous_model=None, previous_date=None):
    """Uniform raw-price fit with update-0 warm start eligible for selection."""
    path = checkpoint_file(run_tag)
    set_seed(SEED + cfg.seed_offset)

    model = MaturityHeadNeuralLocalVol(ctx.maturity_boundaries).to(DEVICE)

    # Completed checkpoints: load model only, without instantiating the large hedge/optimizers.
    if path.exists() and not FORCE_RETRAIN:
        p = safe_torch_load(path)
        if p.get("completed", False):
            model.load_state_dict(p["final_model_state"], strict=True)
            model.eval()
            set_requires_grad(model, False)
            history = pd.DataFrame(p.get("history", []))
            rerank = pd.DataFrame(p.get("rerank_records") or [])
            reason = p.get("stop_reason")
            del p
            gc.collect()
            print("Loaded completed final-v7 checkpoint:", path)
            return model, history, rerank, reason
        del p
        gc.collect()

    hedge = IntervalHedgeBank(ctx.maturity_boundaries, len(ctx.contracts)).to(DEVICE)
    opt_sde = torch.optim.Adam(model.parameters(), lr=cfg.sde_lr)
    opt_hedge = torch.optim.Adam(hedge.parameters(), lr=cfg.hedge_lr)

    next_sde, history, candidates = 0, [], []
    best_monitor, bad_sde = np.inf, 0
    stop_reason = None
    resumed = False

    if path.exists() and not FORCE_RETRAIN:
        p = safe_torch_load(path)
        model.load_state_dict(p["model_state"], strict=True)
        hedge.load_state_dict(p["hedge_state"], strict=True)
        opt_sde.load_state_dict(p["opt_sde"])
        opt_hedge.load_state_dict(p["opt_hedge"])
        next_sde = int(p.get("next_sde_update", 0))
        history = list(p.get("history", []))
        candidates = list(p.get("candidates", []))
        best_monitor = float(p.get("best_monitor", np.inf))
        bad_sde = int(p.get("bad_sde", 0))
        stop_reason = p.get("stop_reason")
        resumed = True
        print(f"Resuming {run_tag} at SDE update {next_sde+1}")
        del p
        gc.collect()
    elif previous_model is not None:
        if previous_date is None:
            raise ValueError("previous_date is required with previous_model.")
        transfer_rolling_model(previous_model, model, previous_date, ctx.date)

    sampler = UniformContractSampler(ctx, cfg.contracts_per_step, SEED + 10 + cfg.seed_offset)
    z_monitor = antithetic_normals(
        cfg.monitor_paths, len(ctx.grid_np)-1, SEED + 100 + cfg.seed_offset
    )

    # The inherited state must be allowed to win. This fixes the earlier polish deterioration.
    if not resumed:
        set_requires_grad(model, False)
        set_requires_grad(hedge, False)
        d0 = objective_diagnostics(model, ctx, z_monitor)
        baseline_score = float(d0["raw_rmse"])
        candidates = update_candidates(model, [], 0, baseline_score, cfg.top_k)
        best_monitor = baseline_score
        history.append({
            "sde_update": 0,
            "sde_train_loss": np.nan,
            "hedge_train_loss": np.nan,
            "monitor_raw_rmse": baseline_score,
            "best_monitor_raw_rmse": baseline_score,
            "bad_sde": 0,
            "sde_lr": cfg.sde_lr,
            "hedge_lr": cfg.hedge_lr,
        })
        print(f"Warm-start raw RMSE before updates: {baseline_score:.4f}")

    bar = tqdm(
        range(next_sde, cfg.max_sde_updates),
        desc=f"v7-final-safe {run_tag}",
        unit="SDE", dynamic_ncols=True, mininterval=1.0,
    )

    for sde_zero in bar:
        sde_update = sde_zero + 1
        sde_loss = run_training_phase(model, hedge, ctx, sampler, opt_sde, "sde", cfg)

        set_requires_grad(model, False)
        set_requires_grad(hedge, False)
        diag = objective_diagnostics(model, ctx, z_monitor)
        score = float(diag["raw_rmse"])
        candidates = update_candidates(model, candidates, sde_update, score, cfg.top_k)

        if score < best_monitor:
            best_monitor = score
            bad_sde = 0
        else:
            bad_sde += 1

        should_stop = (
            sde_update >= cfg.early_min_sde
            and bad_sde >= cfg.early_patience_sde
        )

        hedge_loss = np.nan
        if not should_stop:
            hedge_loss = run_training_phase(model, hedge, ctx, sampler, opt_hedge, "hedge", cfg)

        history.append({
            "sde_update": sde_update,
            "sde_train_loss": sde_loss,
            "hedge_train_loss": hedge_loss,
            "monitor_raw_rmse": score,
            "best_monitor_raw_rmse": best_monitor,
            "bad_sde": bad_sde,
            "sde_lr": cfg.sde_lr,
            "hedge_lr": cfg.hedge_lr,
        })

        bar.set_postfix({
            "raw_RMSE": f"{score:.3f}",
            "best": f"{best_monitor:.3f}",
            "patience": f"{bad_sde}/{cfg.early_patience_sde}",
        })

        if should_stop:
            stop_reason = (
                f"early_stop: no new fixed-path raw-RMSE best for "
                f"{cfg.early_patience_sde} SDE updates"
            )

        if (sde_update % cfg.save_every_sde == 0
            or should_stop
            or sde_update == cfg.max_sde_updates):
            save_training_state(
                path, sde_update, model, hedge, opt_sde, opt_hedge,
                history, candidates, best_monitor, bad_sde,
                completed=False, stop_reason=stop_reason,
            )

        if should_stop:
            print("Early stopping:", stop_reason, "| SDE update", sde_update)
            break

        gc.collect()

    bar.close()
    stop_reason = stop_reason or "max_sde_updates"
    del z_monitor
    gc.collect()

    rerank_table, best = rerank_candidates(ctx, candidates, cfg)
    print(
        f"Selected SDE update {int(best['sde_update'])}: "
        f"50k raw RMSE={float(rerank_table.iloc[0]['rerank_raw_rmse']):.4f}"
    )

    model.load_state_dict(best["model_state"], strict=True)
    model.eval()
    set_requires_grad(model, False)

    final_model_state = cpu_state_dict(model)
    save_training_state(
        path, int(history[-1]["sde_update"]),
        model, hedge, opt_sde, opt_hedge,
        history, candidates, best_monitor, bad_sde,
        completed=True,
        stop_reason=stop_reason,
        final_model_state=final_model_state,
        rerank_records=rerank_table.to_dict("records"),
    )

    del hedge, opt_sde, opt_hedge, candidates, final_model_state
    gc.collect()
    return model, pd.DataFrame(history), rerank_table, stop_reason



def main_train_config(seed_offset=0):
    return TrainConfig(seed_offset=seed_offset)


def polish_train_config(seed_offset=0):
    return TrainConfig(
        max_sde_updates=POLISH_MAX_SDE_UPDATES,
        early_min_sde=POLISH_EARLY_MIN_SDE_UPDATES,
        early_patience_sde=POLISH_EARLY_PATIENCE_SDE_UPDATES,
        seed_offset=seed_offset,
    )


## 7. Pricing, implied-volatility, delta and future-date evaluation helpers


In [7]:
def tkey(t):
    return round(float(t), 12)


def bs_forward_price(F, K, D, T, sigma, cp="C"):
    F, K, D, T, sigma = map(float, (F, K, D, T, sigma))
    if T <= 0 or sigma <= 0:
        return D*max(F-K, 0.0) if cp == "C" else D*max(K-F, 0.0)
    srt = sigma * math.sqrt(T)
    d1 = (math.log(F/K) + 0.5*srt*srt) / srt
    d2 = d1 - srt
    call = D * (F*float(ndtr(d1)) - K*float(ndtr(d2)))
    return call if cp == "C" else call - D*(F-K)


def implied_vol_from_price(price, F, K, D, T, cp):
    intrinsic = D*max(F-K, 0.0) if cp == "C" else D*max(K-F, 0.0)
    upper = D*F if cp == "C" else D*K
    if not (intrinsic-1e-8 <= price <= upper+1e-8) or T <= 0:
        return np.nan
    try:
        return float(brentq(
            lambda s: bs_forward_price(F, K, D, T, s, cp) - price,
            1e-5, 5.0, maxiter=100
        ))
    except ValueError:
        return np.nan


@torch.no_grad()
def simulate_same_day_x(model, maturities, total_paths, chunk_paths, seed, progress=True):
    """Low-memory antithetic MC with preallocated float32 terminal arrays."""
    if total_paths % 2:
        raise ValueError("total_paths must be even for antithetic evaluation.")

    grid = build_time_grid(maturities)
    maturity_idx = {
        tkey(T): int(nearest_grid_indices(grid, [T])[0])
        for T in np.unique(np.asarray(maturities, float))
    }
    index_to_keys = {}
    for key, idx in maturity_idx.items():
        index_to_keys.setdefault(idx, []).append(key)

    total_pairs = total_paths // 2
    chunk_pairs = max(1, chunk_paths // 2)
    collected = {key: np.empty((total_pairs, 2), dtype=np.float32) for key in maturity_idx}

    gen = torch.Generator(device="cpu")
    gen.manual_seed(seed)
    done = 0
    bar = tqdm(total=total_paths, desc="Same-day MC", unit="path", disable=not progress)

    while done < total_pairs:
        n_pairs = min(chunk_pairs, total_pairs-done)
        z_half = torch.randn(n_pairs, len(grid)-1, generator=gen, dtype=torch.float32)
        z = torch.cat([z_half, -z_half], 0).to(DEVICE)
        x = torch.ones((2*n_pairs, 1), dtype=torch.float32, device=DEVICE)

        for step in range(len(grid)-1):
            t = float(grid[step])
            dt = float(grid[step+1] - grid[step])
            sigma = model(torch.full_like(x, t), x)
            dW = math.sqrt(dt) * z[:, step:step+1]
            a = x * sigma
            x = x + a*dW / (1.0 + torch.abs(a)*math.sqrt(dt))

            if step+1 in index_to_keys:
                arr = x.squeeze(1).detach().cpu().numpy()
                lo, hi = done, done+n_pairs
                for key in index_to_keys[step+1]:
                    collected[key][lo:hi, 0] = arr[:n_pairs]
                    collected[key][lo:hi, 1] = arr[n_pairs:]

        done += n_pairs
        bar.update(2*n_pairs)
        del z_half, z, x
        gc.collect()

    bar.close()
    return collected



def price_rows_same_day(rows, samples_by_T, base_col="mid"):
    out = rows.copy().reset_index(drop=True)
    price = np.empty(len(out))

    for T, idx in out.groupby("T").groups.items():
        pos = np.asarray(list(idx), dtype=int)
        g = out.loc[idx]
        samples = samples_by_T[tkey(T)].reshape(-1)
        payoff = empirical_payoff_mean(
            samples,
            g["K"].to_numpy(float) / g["F"].to_numpy(float),
            g["cp_flag"].to_numpy(),
        )
        price[pos] = g["D"].to_numpy(float) * g["F"].to_numpy(float) * payoff

    out["neural_lv_price"] = price
    out["pricing_error"] = price - out[base_col].to_numpy(float)
    out["inside_spread"] = out["neural_lv_price"].between(out["bid"], out["ask"])
    return out


def add_model_iv(rows, desc="Model IV"):
    out = rows.copy()
    iv = [
        implied_vol_from_price(r.neural_lv_price, r.F, r.K, r.D, r.T, r.cp_flag)
        for r in tqdm(out.itertuples(index=False), total=len(out), desc=desc, unit="opt")
    ]
    out["iv_model"] = iv
    market_col = "iv_market" if "iv_market" in out.columns else "iv_market_otm"
    out["iv_error"] = out["iv_model"] - out[market_col].to_numpy(float)
    return out


def delta_rows_from_terminal(rows, samples_by_T, bump_bps=DELTA_BUMP_BPS):
    out = rows.copy().reset_index(drop=True)

    for bps in bump_bps:
        eps = float(bps) * 1e-4
        delta = np.empty(len(out))

        for T, idx in out.groupby("T").groups.items():
            pos = np.asarray(list(idx), dtype=int)
            g = out.loc[idx]
            samples = samples_by_T[tkey(T)].reshape(-1)

            F = g["F"].to_numpy(float)
            K = g["K"].to_numpy(float)
            D = g["D"].to_numpy(float)
            cp = g["cp_flag"].to_numpy()
            S = g["S"].to_numpy(float)

            Fu, Fd = F*(1+eps), F*(1-eps)
            pu = D*Fu*empirical_payoff_mean(samples, K/Fu, cp)
            pdn = D*Fd*empirical_payoff_mean(samples, K/Fd, cp)
            delta[pos] = (pu-pdn) / (2*eps*S)

        out[f"delta_{bps}bp"] = delta

    headline = f"delta_{DELTA_HEADLINE_BPS}bp"
    out["neural_lv_delta"] = out[headline]
    others = [f"delta_{b}bp" for b in bump_bps if b != DELTA_HEADLINE_BPS]
    out["delta_bump_instability"] = np.max(
        np.column_stack([np.abs(out[c]-out[headline]) for c in others]), axis=1
    )
    return out


def price_summary(df, error_col="pricing_error", market_col="mid", inside_col="inside_spread"):
    e = df[error_col].to_numpy(float)
    market = np.maximum(df[market_col].to_numpy(float), 1e-12)

    out = {
        "n": len(df),
        "MSE": float(np.mean(e**2)),
        "RMSE": float(np.sqrt(np.mean(e**2))),
        "MAE": float(np.mean(np.abs(e))),
        "Rel_MAE_pct": float(np.mean(np.abs(e)/market) * 100.0),
    }
    if inside_col in df:
        out["inside_bid_ask"] = float(df[inside_col].mean())
    if "iv_error" in df:
        ie = df["iv_error"].dropna().to_numpy(float)
        out["IV_RMSE"] = float(np.sqrt(np.mean(ie**2))) if len(ie) else np.nan
    return out


def selected_quote_ids(selected):
    return set(selected["optionid"].astype(int))


def evaluate_same_day(model, date, train, holdout, paths=FINAL_MC_PATHS, seed=SEED+7000):
    date = pd.Timestamp(date)
    full = quotes_all.loc[quotes_all["date"] == date].copy().reset_index(drop=True)
    otm_day = otm.loc[otm["date"] == date].rename(columns={
        "mid_otm": "mid", "bid_otm": "bid", "ask_otm": "ask",
        "cp_otm": "cp_flag", "optionid_otm": "optionid",
    }).copy().reset_index(drop=True)

    samples = simulate_same_day_x(
        model, full["T"].unique(), paths, FINAL_PATH_CHUNK, seed
    )
    full_eval = delta_rows_from_terminal(price_rows_same_day(full, samples), samples)
    otm_eval = add_model_iv(price_rows_same_day(otm_day, samples), f"{date.date()} full OTM IV")

    train_eval = add_model_iv(
        full_eval.loc[full_eval["optionid"].astype(int).isin(selected_quote_ids(train))].copy(),
        f"{date.date()} train IV",
    )
    holdout_eval = add_model_iv(
        full_eval.loc[full_eval["optionid"].astype(int).isin(selected_quote_ids(holdout))].copy(),
        f"{date.date()} holdout IV",
    )

    hierarchy = pd.DataFrame({
        "selected_train_80": price_summary(train_eval),
        "selected_holdout_20": price_summary(holdout_eval),
        "full_OTM_strip": price_summary(otm_eval),
        "complete_clean_chain": price_summary(full_eval),
    }).T

    del samples
    gc.collect()
    return hierarchy, full_eval, train_eval, holdout_eval, otm_eval


def evaluate_transition_state(model, date, paths=FINAL_MC_PATHS, seed=SEED+7500):
    """Current-day full-chain prices/deltas for the model actually used to forecast tomorrow."""
    date = pd.Timestamp(date)
    full = quotes_all.loc[quotes_all["date"] == date].copy().reset_index(drop=True)
    samples = simulate_same_day_x(
        model, full["T"].unique(), paths, FINAL_PATH_CHUNK, seed
    )
    out = delta_rows_from_terminal(price_rows_same_day(full, samples), samples)
    del samples
    gc.collect()
    return out


def build_evaluation_pairs(quotes):
    dates = sorted(quotes["date"].unique())
    next_map = {dates[i]: dates[i+1] for i in range(len(dates)-1)}
    left = quotes.copy()
    left["next_date"] = left["date"].map(next_map)

    cols = [
        "date", "optionid", "cp_flag", "K", "exdate", "am_settlement",
        "mid", "bid", "ask", "S", "F", "D", "r", "q", "carry", "T", "k",
        "delta_market",
    ]
    right = quotes[cols].rename(columns={
        "date": "next_date", "mid": "mid_next", "bid": "bid_next",
        "ask": "ask_next", "S": "S_next", "F": "F_next", "D": "D_next",
        "r": "r_next", "q": "q_next", "carry": "carry_next",
        "T": "T_next", "k": "k_next", "delta_market": "delta_market_next",
    })
    return left.merge(
        right,
        on=["next_date", "optionid", "cp_flag", "K", "exdate", "am_settlement"],
        how="inner",
        validate="many_to_one",
    )


@torch.no_grad()
def simulate_frozen_to_date(model, curve0, curve1, elapsed, maturities_remaining,
                            total_paths, chunk_paths, seed, progress=True):
    """Low-memory future-date MC with direct writes into preallocated float32 arrays."""
    if total_paths % 2:
        raise ValueError("total_paths must be even.")

    grid = build_time_grid(maturities_remaining)
    maturity_idx = {
        tkey(T): int(nearest_grid_indices(grid, [T])[0])
        for T in np.unique(np.asarray(maturities_remaining, float))
    }
    index_to_keys = {}
    for key, idx in maturity_idx.items():
        index_to_keys.setdefault(idx, []).append(key)

    total_pairs = total_paths // 2
    chunk_pairs = max(1, chunk_paths // 2)
    collected = {key: np.empty((total_pairs, 2), dtype=np.float32) for key in maturity_idx}

    gen = torch.Generator(device="cpu")
    gen.manual_seed(seed)
    done = 0
    bar = tqdm(
        total=total_paths,
        desc=f"Future-date MC AM={curve1.am_settlement}",
        unit="path",
        disable=not progress,
    )

    while done < total_pairs:
        n_pairs = min(chunk_pairs, total_pairs-done)
        z_half = torch.randn(n_pairs, len(grid)-1, generator=gen, dtype=torch.float32)
        z = torch.cat([z_half, -z_half], 0).to(DEVICE)
        y = torch.ones((2*n_pairs, 1), dtype=torch.float32, device=DEVICE)

        for step in range(len(grid)-1):
            v = float(grid[step])
            dt = float(grid[step+1] - grid[step])
            abs_t = elapsed + v

            Fnew = float(curve1.forward(v))
            Fold = float(curve0.forward(abs_t))
            x_old = y * (Fnew/Fold)

            sigma = model(torch.full_like(y, abs_t), x_old)
            dW = math.sqrt(dt) * z[:, step:step+1]
            a = y * sigma
            y = y + a*dW / (1.0 + torch.abs(a)*math.sqrt(dt))

            if step+1 in index_to_keys:
                arr = y.squeeze(1).detach().cpu().numpy()
                lo, hi = done, done+n_pairs
                for key in index_to_keys[step+1]:
                    collected[key][lo:hi, 0] = arr[:n_pairs]
                    collected[key][lo:hi, 1] = arr[n_pairs:]

        done += n_pairs
        bar.update(2*n_pairs)
        del z_half, z, y
        gc.collect()

    bar.close()
    return collected



def price_future_rows(rows, samples_by_am, price_col="model_price_future"):
    out = rows.copy().reset_index(drop=True)
    price = np.empty(len(out))

    for (am, T), idx in out.groupby(["am_settlement", "T"]).groups.items():
        pos = np.asarray(list(idx), dtype=int)
        g = out.loc[idx]
        samples = samples_by_am[int(am)][tkey(T)].reshape(-1)
        payoff = empirical_payoff_mean(
            samples,
            g["K"].to_numpy(float) / g["F"].to_numpy(float),
            g["cp_flag"].to_numpy(),
        )
        price[pos] = g["D"].to_numpy(float) * g["F"].to_numpy(float) * payoff

    out[price_col] = price
    out["pricing_error"] = price - out["mid"].to_numpy(float)
    out["inside_spread"] = out[price_col].between(out["bid"], out["ask"])
    return out


def evaluate_frozen_on_rows(model, model_date, target_date, rows, paths, seed):
    """Evaluate one settlement group at a time."""
    model_date, target_date = pd.Timestamp(model_date), pd.Timestamp(target_date)
    elapsed = (target_date-model_date).days / 365.0
    pieces = []

    for am in sorted(rows["am_settlement"].unique()):
        g = rows.loc[rows["am_settlement"] == am].copy().reset_index(drop=True)
        samples = simulate_frozen_to_date(
            model,
            curve_book.get(model_date, int(am)),
            curve_book.get(target_date, int(am)),
            elapsed,
            g["T"].unique(),
            paths,
            FINAL_PATH_CHUNK,
            seed + int(am),
        )
        priced = price_future_rows(g, {int(am): samples})
        pieces.append(priced)
        del samples, priced, g
        gc.collect()

    return pd.concat(pieces, ignore_index=True)



def pearson_reduction(x, y):
    """Pearson correlation via elementwise reductions; avoids the corrcoef/dot route.

    This is a diagnostic workaround, not evidence that BLAS caused the crash.
    Inputs are paired after dropping NaNs, as in the existing summary.
    """
    x, y = np.asarray(x, dtype=np.float64), np.asarray(y, dtype=np.float64)
    if x.size < 2 or not (np.isfinite(x).all() and np.isfinite(y).all()):
        return np.nan
    x, y = x - x.mean(), y - y.mean()
    sx, sy = float(np.max(np.abs(x))), float(np.max(np.abs(y)))
    if sx == 0 or sy == 0:
        return np.nan
    x, y = x / sx, y / sy
    denominator = math.sqrt(float(np.sum(x*x))) * math.sqrt(float(np.sum(y*y)))
    return float(np.clip(np.sum(x*y) / denominator, -1.0, 1.0))


def _transition_subset_stats(df):
    if len(df) == 0:
        return {
            "n": 0,
            "pricing_RMSE": np.nan, "pricing_MAE": np.nan, "pricing_Rel_MAE_pct": np.nan,
            "pricing_inside_bid_ask": np.nan,
            "NLV_hedge_RMSE": np.nan, "NLV_hedge_MAE": np.nan, "NLV_hedge_bias": np.nan,
            "market_delta_hedge_RMSE": np.nan, "market_delta_hedge_MAE": np.nan,
            "market_delta_hedge_bias": np.nan,
            "NLV_minus_market_delta_RMSE": np.nan, "delta_correlation": np.nan,
        }

    pe = df["next_day_pricing_error"].to_numpy(float)
    nlv_he = df["hedge_error"].dropna().to_numpy(float)
    mkt_he = df["market_delta_hedge_error"].dropna().to_numpy(float)

    good_delta = df[["neural_lv_delta", "delta_market"]].dropna()
    dd = (good_delta["neural_lv_delta"] - good_delta["delta_market"]).to_numpy(float)

    return {
        "n": len(df),
        "pricing_RMSE": float(np.sqrt(np.mean(pe**2))),
        "pricing_MAE": float(np.mean(np.abs(pe))),
        "pricing_Rel_MAE_pct": float(
            np.mean(np.abs(pe)/np.maximum(df["mid_next"].to_numpy(float), 1e-12))*100
        ),
        "pricing_inside_bid_ask": float(df["next_day_inside_bid_ask"].mean()),
        "NLV_hedge_RMSE": float(np.sqrt(np.mean(nlv_he**2))) if len(nlv_he) else np.nan,
        "NLV_hedge_MAE": float(np.mean(np.abs(nlv_he))) if len(nlv_he) else np.nan,
        "NLV_hedge_bias": float(np.mean(nlv_he)) if len(nlv_he) else np.nan,
        "market_delta_hedge_RMSE": float(np.sqrt(np.mean(mkt_he**2))) if len(mkt_he) else np.nan,
        "market_delta_hedge_MAE": float(np.mean(np.abs(mkt_he))) if len(mkt_he) else np.nan,
        "market_delta_hedge_bias": float(np.mean(mkt_he)) if len(mkt_he) else np.nan,
        "NLV_minus_market_delta_RMSE": float(np.sqrt(np.mean(dd**2))) if len(dd) else np.nan,
        "delta_correlation": pearson_reduction(good_delta["neural_lv_delta"], good_delta["delta_market"])
            if len(good_delta) > 1 else np.nan,
    }


def evaluate_transition(model, date, next_date, same_day_full, paths=FINAL_MC_PATHS, seed=SEED+8000):
    """Low-memory next-day evaluation: one settlement group at a time."""
    stage_log("transition: begin")
    date, next_date = pd.Timestamp(date), pd.Timestamp(next_date)
    pairs = evaluation_pairs.loc[
        (evaluation_pairs["date"] == date) & (evaluation_pairs["next_date"] == next_date)
    ].copy().reset_index(drop=True)

    elapsed = (next_date-date).days / 365.0
    priced_pieces = []

    for am in sorted(pairs["am_settlement"].unique()):
        g = pairs.loc[pairs["am_settlement"] == am].copy().reset_index(drop=True)
        stage_log(f"transition AM={am}: simulation begin, rows={len(g)}")
        samples = simulate_frozen_to_date(
            model,
            curve_book.get(date, int(am)),
            curve_book.get(next_date, int(am)),
            elapsed,
            g["T_next"].unique(),
            paths,
            FINAL_PATH_CHUNK,
            seed + int(am),
        )

        stage_log(f"transition AM={am}: simulation returned; terminal storage={sum(v.nbytes for v in samples.values())/2**20:.1f} MiB")
        price = np.empty(len(g), dtype=np.float64)
        for T, idx in g.groupby("T_next").groups.items():
            pos = np.asarray(list(idx), dtype=int)
            gg = g.loc[idx]
            terminal = samples[tkey(T)].reshape(-1)
            stage_log(f"transition AM={am}: payoff begin T={T}, contracts={len(gg)}")
            payoff = empirical_payoff_mean(
                terminal,
                gg["K"].to_numpy(float) / gg["F_next"].to_numpy(float),
                gg["cp_flag"].to_numpy(),
            )
            price[pos] = gg["D_next"].to_numpy(float) * gg["F_next"].to_numpy(float) * payoff

        stage_log(f"transition AM={am}: pricing complete")
        del terminal, gg, payoff
        g["model_price_next"] = price
        priced_pieces.append(g)
        del samples, price, g
        gc.collect()

    stage_log("transition: concatenate begin")
    out = pd.concat(priced_pieces, ignore_index=True)
    del priced_pieces, pairs
    gc.collect()

    out["next_day_pricing_error"] = out["model_price_next"] - out["mid_next"].to_numpy(float)
    out["next_day_inside_bid_ask"] = out["model_price_next"].between(out["bid_next"], out["ask_next"])

    stage_log("transition: delta merge begin")
    delta_map = same_day_full[
        ["optionid", "neural_lv_delta", "delta_bump_instability"]
    ].drop_duplicates("optionid")
    out = out.merge(delta_map, on="optionid", how="left", validate="many_to_one")
    out["dP_market"] = out["mid_next"] - out["mid"]
    out["dS"] = out["S_next"] - out["S"]
    out["hedge_error"] = out["dP_market"] - out["neural_lv_delta"]*out["dS"]
    out["market_delta_hedge_error"] = out["dP_market"] - out["delta_market"]*out["dS"]

    stage_log("transition: hedge calculations complete; OTM flags begin")
    current_otm_ids = otm_option_ids_by_date.get(date, set())
    next_otm_ids = otm_option_ids_by_date.get(next_date, set())
    ids = out["optionid"].astype(int)
    out["current_is_otm"] = ids.isin(current_otm_ids)
    out["next_is_otm"] = ids.isin(next_otm_ids)
    out["otm_on_both_dates"] = out["current_is_otm"] & out["next_is_otm"]

    stage_log("transition: full summary begin")
    full_stats = _transition_subset_stats(out)
    stage_log("transition: OTM summary begin")
    otm_stats = _transition_subset_stats(out.loc[out["otm_on_both_dates"]].copy())

    summary = {"date": date, "next_date": next_date}
    summary.update({f"full_{k}": v for k, v in full_stats.items()})
    summary.update({f"otm_{k}": v for k, v in otm_stats.items()})
    gc.collect()
    stage_log("transition: complete")
    return summary, out



def bs_delta_unit_test():
    class ConstantVol(nn.Module):
        def __init__(self, sigma):
            super().__init__()
            self.sigma = float(sigma)
        def forward(self, t, x):
            return torch.full_like(x, self.sigma)

    S, r, q, T, K, sig = 5000.0, 0.04, 0.015, 0.5, 5100.0, 0.20
    F = S*math.exp((r-q)*T)
    D = math.exp(-r*T)

    samples = simulate_same_day_x(
        ConstantVol(sig).to(DEVICE), [T], 30_000, 10_000, SEED+6600, progress=False
    )
    row = pd.DataFrame([{"S": S, "F": F, "D": D, "K": K, "T": T, "cp_flag": "C"}])
    mc = float(delta_rows_from_terminal(row, samples)["neural_lv_delta"].iloc[0])

    d1 = (math.log(S/K)+(r-q+0.5*sig*sig)*T)/(sig*math.sqrt(T))
    exact = math.exp(-q*T)*float(ndtr(d1))
    err = abs(mc-exact)

    print(f"BS delta unit test: MC={mc:.5f}, analytic={exact:.5f}, abs error={err:.5f}")
    if err > 0.03:
        warnings.warn("Delta unit test is noisier than expected.")
    return err


def clock_unit_test():
    class ToyCurve:
        am_settlement = 0
        def __init__(self, spot, carry):
            self.spot, self.carry = float(spot), float(carry)
        def forward(self, t):
            return self.spot*np.exp(self.carry*np.asarray(t, float))

    class Recorder(nn.Module):
        def __init__(self):
            super().__init__()
            self.times = []
        def forward(self, t, x):
            self.times.append(float(t.reshape(-1)[0]))
            return torch.full_like(x, 0.2)

    elapsed = 0.10
    rec = Recorder().to(DEVICE)
    simulate_frozen_to_date(
        rec, ToyCurve(100, 0.02), ToyCurve(101, 0.02),
        elapsed, [0.08], 4, 4, SEED+6700, progress=False
    )
    err = abs(rec.times[0] - elapsed)
    print(f"Clock unit test error={err:.2e}")
    if err > 1e-6:
        raise RuntimeError("Frozen model clock restarted incorrectly.")
    return err


otm_option_ids_by_date = {
    pd.Timestamp(d): set(g["optionid_otm"].dropna().astype(int))
    for d, g in otm.groupby("date")
}
evaluation_pairs = build_evaluation_pairs(quotes_all)


## 8. March 1 anchor — legacy validation, then 100% raw forecast polish

The inherited v6 raw-loss maturity-head model is first evaluated on the new all-OTM 80/20 partition. This remains an **anchor validation/audit** rather than a pristine test.

After that measurement, March 1 is polished on **100% of the filtered OTM surface** using the final raw trainer. The polished model is the one used for Mar-1→Mar-4 forecasting, the frozen-anchor comparator, and the warm start into Mar 4.


In [8]:
def load_v6_anchor_model(ctx):
    path = locate_v6_anchor(ctx.date)
    if path is None:
        raise FileNotFoundError(
            "Completed v6 raw-loss maturity-head checkpoint not found under neural_lv_v6_cache."
        )

    payload = safe_torch_load(path)
    model = MaturityHeadNeuralLocalVol(ctx.maturity_boundaries).to(DEVICE)
    model_state = payload.get("final_model_state") or payload.get("model_state")
    model.load_state_dict(model_state, strict=True)
    model.eval()
    set_requires_grad(model, False)
    del payload
    gc.collect()
    print("Loaded v6 anchor model:", path)
    return model, path


delta_unit_test_error = bs_delta_unit_test()
clock_unit_test_error = clock_unit_test()
trunk_shift_unit_test_error = trunk_time_shift_unit_test()

anchor_ctx_80 = build_calibration_context(ANCHOR_DATE, anchor_train)
anchor_model_80, anchor_checkpoint = load_v6_anchor_model(anchor_ctx_80)

anchor_hierarchy, anchor_full_80, anchor_train_eval, anchor_validation_eval, anchor_otm_eval = evaluate_same_day(
    anchor_model_80, ANCHOR_DATE, anchor_train, anchor_holdout,
    paths=FINAL_MC_PATHS, seed=SEED+7000
)

print("\nMarch 1 inherited-anchor evaluation (100k paths)")
print(anchor_hierarchy.to_string())
print("Median delta bump instability:", float(anchor_full_80["delta_bump_instability"].median()))

# After measuring the hold-out, use every current-day OTM quote for the forecasting model.
anchor_ctx_full = build_calibration_context(ANCHOR_DATE, anchor_selected)
anchor_forecast_model, anchor_polish_hist, anchor_polish_rerank, anchor_polish_stop = train_raw_day(
    anchor_ctx_full,
    polish_train_config(seed_offset=50_000),
    "forecast_full_20240301",
    previous_model=anchor_model_80,
    previous_date=ANCHOR_DATE,
)

anchor_forecast_state = evaluate_transition_state(
    anchor_forecast_model, ANCHOR_DATE,
    paths=FINAL_MC_PATHS, seed=SEED+7500
)
anchor_forecast_otm = anchor_forecast_state.loc[
    anchor_forecast_state["optionid"].astype(int).isin(otm_option_ids_by_date[ANCHOR_DATE])
].copy()
anchor_forecast_fit = price_summary(anchor_forecast_otm)

print("\nMarch 1 100%-OTM forecast-model same-day fit")
print(pd.Series(anchor_forecast_fit).to_string())
print("Anchor polish stop:", anchor_polish_stop)

anchor_hierarchy.to_csv(CACHE_DIR / "anchor_20240301_validation.csv")
anchor_validation_eval.to_csv(CACHE_DIR / "anchor_20240301_validation_contracts.csv", index=False)
anchor_polish_hist.to_csv(CACHE_DIR / "anchor_20240301_polish_history.csv", index=False)
anchor_polish_rerank.to_csv(CACHE_DIR / "anchor_20240301_polish_rerank.csv", index=False)

# Preserve only the scalar needed later, then release large/unused anchor objects.
anchor_delta_bump_instability = float(anchor_full_80["delta_bump_instability"].median())
del anchor_ctx_80, anchor_ctx_full, anchor_model_80
del anchor_full_80, anchor_train_eval, anchor_otm_eval
gc.collect()
print("Released nonessential Section-8 anchor objects before rolling evaluation.")


BS delta unit test: MC=0.50156, analytic=0.50382, abs error=0.00226
Clock unit test error=1.49e-09
Warm-start 2024-03-01 -> 2024-03-04: 3 heads | elapsed=0.00822y | median head mismatch=0.00178y
Trunk time-shift unit test max error=3.58e-07
Loaded v6 anchor model: neural_lv_v6_cache\real_20240301_split80_20_ablation_rawloss_maturity_heads_resume.pt


2024-03-01 holdout IV: 100%|██████████| 861/861 [00:00<00:00, 16380.28opt/s]



March 1 inherited-anchor evaluation (100k paths)
                           n        MSE      RMSE       MAE  Rel_MAE_pct  inside_bid_ask   IV_RMSE
selected_train_80     3443.0   2.768645  1.663925  1.167055     3.485000        0.152483  0.002945
selected_holdout_20    861.0   2.659872  1.630911  1.162621     3.539421        0.152149  0.002955
full_OTM_strip        4304.0   2.746885  1.657373  1.166168     3.495887        0.152416  0.002947
complete_clean_chain  9046.0  33.316739  5.772065  4.242764     2.812710        0.128344       NaN
Median delta bump instability: 3.338957239430729e-05
Loaded completed final-v7 checkpoint: neural_lv_v7_final_safe_cache\forecast_full_20240301_resume.pt


Same-day MC: 100%|██████████| 100000/100000 [00:08<00:00, 11652.35path/s]



March 1 100%-OTM forecast-model same-day fit
n                 4304.000000
MSE                  2.922486
RMSE                 1.709528
MAE                  1.159305
Rel_MAE_pct          3.628639
inside_bid_ask       0.145214
Anchor polish stop: early_stop: no new fixed-path raw-RMSE best for 12 SDE updates
Released nonessential Section-8 anchor objects before rolling evaluation.


## 9. Main experiment — clean same-day hold-out, full-data forecast model, next-day test

For each later calibration date:

1. warm-start from the **previous day's 100%-OTM forecast model** with the corrected calendar-time transfer;
2. fit only today's 80% training subset using plain raw contract-level MSE;
3. evaluate the untouched 20% same-day hold-out;
4. warm-start from that fitted model and polish on **100% of today's filtered OTM observations**;
5. use the full-data model for today's deltas, next-day pricing, and tomorrow's warm start;
6. report next-day **OTM-on-both-dates** metrics as primary, with full-chain metrics as secondary;
7. compare Neural-LV delta hedging to the OptionMetrics market-delta diagnostic on exactly the same contracts.


In [9]:
stage_log("Section 9 begin")
try:
    anchor_index = trading_dates.index(ANCHOR_DATE)
    available_transitions = len(trading_dates) - anchor_index - 1
    n_transitions = available_transitions if MAX_TRANSITIONS is None else min(MAX_TRANSITIONS, available_transitions)
    sequence = trading_dates[anchor_index:anchor_index+n_transitions+1]
    
    print("Rolling sequence:", " -> ".join(str(pd.Timestamp(d).date()) for d in sequence))
    
    daily_records = []
    transition_records = []
    same_day_exports = []
    transition_exports = []
    
    # Anchor validation row.
    a_hold = anchor_hierarchy.loc["selected_holdout_20"]
    daily_records.append({
        "date": ANCHOR_DATE,
        "role": "anchor_validation",
        "train_RMSE": float(anchor_hierarchy.loc["selected_train_80", "RMSE"]),
        "holdout_RMSE": float(a_hold["RMSE"]),
        "holdout_MAE": float(a_hold["MAE"]),
        "holdout_IV_RMSE": float(a_hold["IV_RMSE"]),
        "holdout_inside_bid_ask": float(a_hold["inside_bid_ask"]),
        "full_OTM_RMSE": float(anchor_hierarchy.loc["full_OTM_strip", "RMSE"]),
        "frozen_anchor_holdout_RMSE": float(a_hold["RMSE"]),
        "fit_stop_reason": "loaded_v6_anchor",
        "fit_sde_updates": 0,
        "polish_stop_reason": anchor_polish_stop,
        "polish_sde_updates": int(anchor_polish_hist["sde_update"].max()),
    })
    same_day_exports.append(anchor_validation_eval.assign(date=ANCHOR_DATE, split="anchor_validation"))
    
    # First transition uses the 100%-OTM-polished anchor model and its own deltas.
    if n_transitions > 0:
        s, t = sequence[0], sequence[1]
        ts, tr = evaluate_transition(
            anchor_forecast_model, s, t, anchor_forecast_state,
            paths=FINAL_MC_PATHS, seed=SEED+8000
        )
        stage_log("first transition returned; saving contract results")
        tr.to_csv(CACHE_DIR / "v7_first_transition_contract_results.csv", index=False)
        pd.DataFrame([ts]).to_csv(CACHE_DIR / "v7_first_transition_summary.csv", index=False)
        transition_records.append(ts)
        transition_exports.append(tr)
    
    previous_forecast_model = anchor_forecast_model
    previous_date = ANCHOR_DATE
    
    # The final sequence date is only the target of the last transition.
    for i, date in enumerate(sequence[1:-1], start=1):
        date = pd.Timestamp(date)
        next_date = pd.Timestamp(sequence[i+1])
    
        print("\n" + "="*88)
        print("ROLLING FINAL-v7:", date.date(), "->", next_date.date())
    
        selected, train, holdout = daily_split(date)
    
        # ---------- A. 80% model for clean same-day interpolation ----------
        ctx80 = build_calibration_context(date, train)
        fit_model, fit_hist, fit_rerank, fit_stop = train_raw_day(
            ctx80,
            main_train_config(seed_offset=1000*i),
            f"fit80_{date:%Y%m%d}",
            previous_model=previous_forecast_model,
            previous_date=previous_date,
        )
    
        hierarchy, fit_full_eval, train_eval, holdout_eval, otm_eval = evaluate_same_day(
            fit_model, date, train, holdout,
            paths=FINAL_MC_PATHS, seed=SEED+7000+i*100
        )
    
        # Frozen-anchor comparator uses the fairer 100%-OTM March-1 forecasting model.
        target_rows = quotes_all.loc[
            (quotes_all["date"] == date)
            & (quotes_all["optionid"].astype(int).isin(selected_quote_ids(holdout)))
        ].copy().reset_index(drop=True)
    
        frozen_eval = evaluate_frozen_on_rows(
            anchor_forecast_model, ANCHOR_DATE, date, target_rows,
            FINAL_MC_PATHS, SEED+12000+i*100
        )
        frozen_summary = price_summary(frozen_eval)
    
        h = hierarchy.loc["selected_holdout_20"]
    
        # ---------- B. 100% current-day polish for the actual forecast model ----------
        ctx100 = build_calibration_context(date, selected)
        forecast_model, polish_hist, polish_rerank, polish_stop = train_raw_day(
            ctx100,
            polish_train_config(seed_offset=20_000+i*1000),
            f"forecast_full_{date:%Y%m%d}",
            previous_model=fit_model,
            previous_date=date,      # elapsed=0: direct same-day weight transfer
        )
    
        forecast_state = evaluate_transition_state(
            forecast_model, date,
            paths=FINAL_MC_PATHS, seed=SEED+7500+i*100
        )
    
        daily_records.append({
            "date": date,
            "role": "clean_future_holdout",
            "train_RMSE": float(hierarchy.loc["selected_train_80", "RMSE"]),
            "holdout_RMSE": float(h["RMSE"]),
            "holdout_MAE": float(h["MAE"]),
            "holdout_IV_RMSE": float(h["IV_RMSE"]),
            "holdout_inside_bid_ask": float(h["inside_bid_ask"]),
            "full_OTM_RMSE": float(hierarchy.loc["full_OTM_strip", "RMSE"]),
            "frozen_anchor_holdout_RMSE": float(frozen_summary["RMSE"]),
            "fit_stop_reason": fit_stop,
            "fit_sde_updates": int(fit_hist["sde_update"].max()),
            "polish_stop_reason": polish_stop,
            "polish_sde_updates": int(polish_hist["sde_update"].max()),
        })
        same_day_exports.append(holdout_eval.assign(date=date, split="clean_future_holdout"))
    
        ts, tr = evaluate_transition(
            forecast_model, date, next_date, forecast_state,
            paths=FINAL_MC_PATHS, seed=SEED+8000+i*100
        )
        transition_records.append(ts)
        transition_exports.append(tr)
    
        # Tomorrow starts from today's model that has seen every current-day OTM quote.
        previous_forecast_model = forecast_model
        previous_date = date
    
        # Persist compact per-date optimization diagnostics.
        fit_hist.to_csv(CACHE_DIR / f"fit80_{date:%Y%m%d}_history.csv", index=False)
        fit_rerank.to_csv(CACHE_DIR / f"fit80_{date:%Y%m%d}_rerank.csv", index=False)
        polish_hist.to_csv(CACHE_DIR / f"forecast_full_{date:%Y%m%d}_history.csv", index=False)
        polish_rerank.to_csv(CACHE_DIR / f"forecast_full_{date:%Y%m%d}_rerank.csv", index=False)
    
        del ctx80, ctx100, fit_model, fit_full_eval, train_eval, holdout_eval, otm_eval
        del frozen_eval, forecast_state
        gc.collect()
    
    
    daily_summary = pd.DataFrame(daily_records)
    transition_summary = pd.DataFrame(transition_records)
    
    print("\nDAILY SAME-DAY INTERPOLATION SUMMARY")
    print(daily_summary.to_string(index=False))
    
    print("\nNEXT-DAY SUMMARY — OTM ON BOTH DATES IS PRIMARY")
    primary_cols = [
        "date", "next_date", "otm_n", "otm_pricing_RMSE", "otm_pricing_MAE",
        "otm_pricing_Rel_MAE_pct", "otm_pricing_inside_bid_ask",
        "otm_NLV_hedge_RMSE", "otm_NLV_hedge_MAE", "otm_NLV_hedge_bias",
        "otm_market_delta_hedge_RMSE", "otm_market_delta_hedge_MAE",
        "otm_market_delta_hedge_bias", "otm_NLV_minus_market_delta_RMSE",
        "otm_delta_correlation",
    ]
    print(transition_summary[primary_cols].to_string(index=False))
    
    print("\nSECONDARY FULL-CLEAN-CHAIN NEXT-DAY PRICING")
    secondary_cols = [
        "date", "next_date", "full_n", "full_pricing_RMSE", "full_pricing_MAE",
        "full_pricing_Rel_MAE_pct", "full_pricing_inside_bid_ask",
    ]
    print(transition_summary[secondary_cols].to_string(index=False))
    
    daily_summary.to_csv(CACHE_DIR / "v7_final_daily_summary.csv", index=False)
    transition_summary.to_csv(CACHE_DIR / "v7_final_transition_summary.csv", index=False)
    
    if same_day_exports:
        pd.concat(same_day_exports, ignore_index=True).to_csv(
            CACHE_DIR / "v7_final_holdout_contract_results.csv", index=False
        )
    if transition_exports:
        pd.concat(transition_exports, ignore_index=True).to_csv(
            CACHE_DIR / "v7_final_next_day_contract_results.csv", index=False
        )
except Exception:
    with open(CACHE_DIR / "v7_python_error.log", "a", encoding="utf-8") as handle:
        traceback.print_exc(file=handle)
    stage_log("Section 9 raised Python exception; see v7_python_error.log")
    raise


2026-09-21T10:28:50 | Section 9 begin | RSS=592.9 MiB
Rolling sequence: 2024-03-01 -> 2024-03-04 -> 2024-03-05 -> 2024-03-06 -> 2024-03-07 -> 2024-03-08
2026-09-21T10:28:50 | transition: begin | RSS=593.6 MiB
2026-09-21T10:28:50 | transition AM=0: simulation begin, rows=6099 | RSS=601.7 MiB


Future-date MC AM=0: 100%|██████████| 100000/100000 [00:08<00:00, 12285.78path/s]

2026-09-21T10:28:58 | transition AM=0: simulation returned; terminal storage=10.7 MiB | RSS=580.6 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.0410958904109589, contracts=155 | RSS=580.7 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.043835616438356165, contracts=165 | RSS=582.9 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.04657534246575343, contracts=164 | RSS=582.9 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.049315068493150684, contracts=203 | RSS=582.9 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.057534246575342465, contracts=167 | RSS=582.9 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.06027397260273973, contracts=153 | RSS=582.9 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.06301369863013699, contracts=171 | RSS=582.9 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.06575342465753424, contracts=255 | RSS=583.0 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.

2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.0958904109589041, contracts=93 | RSS=583.0 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.09863013698630137, contracts=53 | RSS=583.0 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.10684931506849316, contracts=164 | RSS=583.0 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.12602739726027398, contracts=333 | RSS=583.1 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.14520547945205478, contracts=113 | RSS=583.1 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.15616438356164383, contracts=382 | RSS=583.1 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.20273972602739726, contracts=384 | RSS=583.1 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.2410958904109589, contracts=467 | RSS=583.2 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.29863013698630136, contracts=325 | RSS=583.2 MiB
2026-09-21T10:28:58 | transition AM=0: payoff begin T=0.317

Future-date MC AM=1: 100%|██████████| 100000/100000 [00:07<00:00, 12646.61path/s]

...

2026-09-21T14:58:20 | transition AM=0: simulation returned; terminal storage=10.7 MiB | RSS=675.5 MiB
2026-09-21T14:58:20 | transition AM=0: payoff begin T=0.04657534246575343, contracts=173 | RSS=675.6 MiB
2026-09-21T14:58:21 | transition AM=0: payoff begin T=0.049315068493150684, contracts=168 | RSS=677.9 MiB
2026-09-21T14:58:21 | transition AM=0: payoff begin T=0.052054794520547946, contracts=182 | RSS=677.9 MiB
2026-09-21T14:58:21 | transition AM=0: payoff begin T=0.0547945205479452, contracts=236 | RSS=677.9 MiB
2026-09-21T14:58:21 | transition AM=0: payoff begin T=0.06575342465753424, contracts=165 | RSS=677.9 MiB
2026-09-21T14:58:21 | transition AM=0: payoff begin T=0.0684931506849315, contracts=152 | RSS=678.0 MiB
2026-09-21T14:58:21 | transition AM=0: payoff begin T=0.07123287671232877, contracts=144 | RSS=678.0 MiB
2026-09-21T14:58:21 | transition AM=0: payoff begin T=0.07397260273972603, contracts=102 | RSS=678.0 MiB


2026-09-21T14:58:21 | transition AM=0: payoff begin T=0.07671232876712329, contracts=237 | RSS=678.1 MiB
2026-09-21T14:58:21 | transition AM=0: payoff begin T=0.08493150684931507, contracts=107 | RSS=678.1 MiB
2026-09-21T14:58:21 | transition AM=0: payoff begin T=0.08767123287671233, contracts=98 | RSS=678.2 MiB
2026-09-21T14:58:21 | transition AM=0: payoff begin T=0.09041095890410959, contracts=93 | RSS=678.2 MiB
2026-09-21T14:58:21 | transition AM=0: payoff begin T=0.09315068493150686, contracts=66 | RSS=678.2 MiB
2026-09-21T14:58:21 | transition AM=0: payoff begin T=0.0958904109589041, contracts=253 | RSS=678.2 MiB
2026-09-21T14:58:21 | transition AM=0: payoff begin T=0.10410958904109589, contracts=94 | RSS=678.2 MiB
2026-09-21T14:58:21 | transition AM=0: payoff begin T=0.11506849315068493, contracts=329 | RSS=678.3 MiB
2026-09-21T14:58:21 | transition AM=0: payoff begin T=0.13424657534246576, contracts=174 | RSS=678.3 MiB
2026-09-21T14:58:21 | transition AM=0: payoff begin T=0.1452

Future-date MC AM=1: 100%|██████████| 100000/100000 [00:10<00:00, 9438.65path/s] 

2026-09-21T14:58:32 | transition AM=1: simulation returned; terminal storage=4.2 MiB | RSS=669.8 MiB
2026-09-21T14:58:32 | transition AM=1: payoff begin T=0.11506849315068493, contracts=330 | RSS=669.9 MiB
2026-09-21T14:58:32 | transition AM=1: payoff begin T=0.1917808219178082, contracts=398 | RSS=672.3 MiB
2026-09-21T14:58:32 | transition AM=1: payoff begin T=0.2876712328767123, contracts=415 | RSS=672.6 MiB
2026-09-21T14:58:32 | transition AM=1: payoff begin T=0.3643835616438356, contracts=358 | RSS=672.6 MiB
2026-09-21T14:58:32 | transition AM=1: payoff begin T=0.4410958904109589, contracts=352 | RSS=672.7 MiB
2026-09-21T14:58:32 | transition AM=1: payoff begin T=0.536986301369863, contracts=146 | RSS=672.7 MiB
2026-09-21T14:58:32 | transition AM=1: payoff begin T=0.6136986301369863, contracts=119 | RSS=672.7 MiB
2026-09-21T14:58:32 | transition AM=1: payoff begin T=0.6904109589041096, contracts=121 | RSS=672.7 MiB
2026-09-21T14:58:32 | transition AM=1: payoff begin T=0.78630136986

2026-09-21T14:58:32 | transition AM=1: payoff begin T=0.863013698630137, contracts=143 | RSS=672.7 MiB
2026-09-21T14:58:32 | transition AM=1: payoff begin T=0.958904109589041, contracts=164 | RSS=672.8 MiB
2026-09-21T14:58:32 | transition AM=1: pricing complete | RSS=672.8 MiB
2026-09-21T14:58:32 | transition: concatenate begin | RSS=672.8 MiB
2026-09-21T14:58:32 | transition: delta merge begin | RSS=670.5 MiB
2026-09-21T14:58:32 | transition: hedge calculations complete; OTM flags begin | RSS=673.1 MiB
2026-09-21T14:58:32 | transition: full summary begin | RSS=673.1 MiB
2026-09-21T14:58:32 | transition: OTM summary begin | RSS=673.1 MiB
2026-09-21T14:58:32 | transition: complete | RSS=673.1 MiB

DAILY SAME-DAY INTERPOLATION SUMMARY
      date                 role  train_RMSE  holdout_RMSE  holdout_MAE  holdout_IV_RMSE  holdout_inside_bid_ask  full_OTM_RMSE  frozen_anchor_holdout_RMSE                                                fit_stop_reason  fit_sde_updates                       

## 10. Compact final NLV summary

In [10]:
future_days = daily_summary.loc[daily_summary["role"] == "clean_future_holdout"].copy()

final_summary = {
    "model": "Neural LV",
    "architecture": "shared_4x50_trunk_plus_maturity_heads",
    "calibration_universe": "all_filtered_OTM" if USE_ALL_FILTERED_OTM else "capped_OTM",
    "loss": "plain_contract_level_raw_price_MSE_spot_scaled",
    "sampler": "uniform_over_training_contracts",
    "checkpoint_metric": "fixed_path_raw_RMSE_every_SDE_update_then_50k_raw_RMSE_rerank",
    "forecast_protocol": "80pct_fit_for_holdout_then_100pct_current_day_OTM_polish",
    "rolling_transfer": "calendar_head_shift_plus_exact_trunk_time_input_shift",
    "anchor_date": str(ANCHOR_DATE.date()),
    "compute_profile": COMPUTE_PROFILE,
    "n_transitions": int(len(transition_summary)),
    "anchor_validation_RMSE_100k": float(
        daily_summary.loc[daily_summary["role"] == "anchor_validation", "holdout_RMSE"].iloc[0]
    ),
    "anchor_100pct_OTM_forecast_fit_RMSE_100k": float(anchor_forecast_fit["RMSE"]),
    "mean_future_holdout_RMSE": float(future_days["holdout_RMSE"].mean()) if len(future_days) else np.nan,
    "mean_frozen_anchor_same_contract_RMSE": float(
        future_days["frozen_anchor_holdout_RMSE"].mean()
    ) if len(future_days) else np.nan,
    "mean_next_day_OTM_pricing_RMSE": float(transition_summary["otm_pricing_RMSE"].mean()),
    "mean_next_day_OTM_pricing_MAE": float(transition_summary["otm_pricing_MAE"].mean()),
    "mean_next_day_full_chain_pricing_RMSE_secondary": float(
        transition_summary["full_pricing_RMSE"].mean()
    ),
    "mean_OTM_NLV_hedge_RMSE": float(transition_summary["otm_NLV_hedge_RMSE"].mean()),
    "mean_OTM_market_delta_hedge_RMSE": float(
        transition_summary["otm_market_delta_hedge_RMSE"].mean()
    ),
    "mean_OTM_NLV_minus_market_delta_RMSE": float(
        transition_summary["otm_NLV_minus_market_delta_RMSE"].mean()
    ),
    "mean_OTM_delta_correlation": float(transition_summary["otm_delta_correlation"].mean()),
    "anchor_delta_bump_instability": float(anchor_delta_bump_instability),
    "BS_delta_unit_test_abs_error": float(delta_unit_test_error),
    "clock_unit_test_abs_error": float(clock_unit_test_error),
    "trunk_time_shift_unit_test_abs_error": float(trunk_shift_unit_test_error),
}

print(json.dumps(final_summary, indent=2))
pd.DataFrame([final_summary]).to_csv(CACHE_DIR / "v7_final_nlv_summary.csv", index=False)
print("\nSaved final-v7 outputs under:", CACHE_DIR.resolve())


{
  "model": "Neural LV",
  "architecture": "shared_4x50_trunk_plus_maturity_heads",
  "calibration_universe": "all_filtered_OTM",
  "loss": "plain_contract_level_raw_price_MSE_spot_scaled",
  "sampler": "uniform_over_training_contracts",
  "checkpoint_metric": "fixed_path_raw_RMSE_every_SDE_update_then_50k_raw_RMSE_rerank",
  "forecast_protocol": "80pct_fit_for_holdout_then_100pct_current_day_OTM_polish",
  "rolling_transfer": "calendar_head_shift_plus_exact_trunk_time_input_shift",
  "anchor_date": "2024-03-01",
  "compute_profile": "cpu",
  "n_transitions": 5,
  "anchor_validation_RMSE_100k": 1.630911370858062,
  "anchor_100pct_OTM_forecast_fit_RMSE_100k": 1.7095280334405907,
  "mean_future_holdout_RMSE": 1.9461554434077302,
  "mean_frozen_anchor_same_contract_RMSE": 4.5739569305778165,
  "mean_next_day_OTM_pricing_RMSE": 4.1189269588714,
  "mean_next_day_OTM_pricing_MAE": 3.491612600505784,
  "mean_next_day_full_chain_pricing_RMSE_secondary": 7.4203846994697615,
  "mean_OTM_NLV_hed

## 11. OPTIONAL — first-future-date stochastic robustness

Leave this off for the main overnight run. If extra compute remains, repeat the first future-date **80% raw fit** three times from the same fixed March-1 100%-OTM forecast model, changing only stochastic seeds.

This tests Monte Carlo/minibatch/optimizer noise. It does not replace genuinely independent cold-start robustness, which is more appropriate if GPU/HPC becomes available.


In [ ]:
if not RUN_SEED_ROBUSTNESS:
    print("Seed robustness skipped.")
elif len(sequence) < 2:
    print("No future date available.")
else:
    seed_date = pd.Timestamp(sequence[1])
    _, seed_train, seed_holdout = daily_split(seed_date)
    seed_ctx = build_calibration_context(seed_date, seed_train)
    seed_rows = []

    for j, offset in enumerate([31_000, 32_000, 33_000], start=1):
        m, hist, rerank, reason = train_raw_day(
            seed_ctx,
            main_train_config(seed_offset=offset),
            f"seed_robust_{seed_date:%Y%m%d}_{j}",
            previous_model=anchor_forecast_model,
            previous_date=ANCHOR_DATE,
        )
        hierarchy, *_ = evaluate_same_day(
            m, seed_date, seed_train, seed_holdout,
            paths=FINAL_MC_PATHS, seed=SEED+40_000+j*100
        )
        seed_rows.append({
            "seed_run": j,
            "heldout_RMSE": float(hierarchy.loc["selected_holdout_20", "RMSE"]),
            "heldout_IV_RMSE": float(hierarchy.loc["selected_holdout_20", "IV_RMSE"]),
            "sde_updates": int(hist["sde_update"].max()),
            "stop_reason": reason,
        })
        del m
        gc.collect()

    seed_robustness = pd.DataFrame(seed_rows)
    print(seed_robustness.to_string(index=False))
    seed_robustness.to_csv(CACHE_DIR / "v7_final_seed_robustness.csv", index=False)


### v7 stopping point

This notebook is intended to freeze Neural LV after the main rolling run.

The next modelling stage is **Neural SLV**. Once NSLV is complete, BS-IVF, classical LV, Heston, NLV and NSLV should be assembled in a separate thin benchmark notebook on common contract sets and common pricing/hedging metrics.
